# Reliable File Ingestion and Data Quality Pipeline

### End-to-End Data Engineering Project

**Technologies:** Python, Pandas, DuckDB, SQL, Jupyter Notebook

---

## 1. Project Overview

This project implements a reliable end-to-end data pipeline
that ingests incoming CSV files, validates records,
handles duplicates and bad records, transforms data,
and publishes clean datasets for analytical consumption.

The pipeline follows a multi-zone architecture:

- Raw Zone
- Validated Zone
- Curated Zone
- Quarantine Zone

It also supports late-arriving data, partitioning,
idempotent processing, logging, monitoring,
and automated testing.

## Import Required Libraries

In [237]:
# Import required libraries

import os
import pandas as pd
import numpy as np
import duckdb
import logging
import hashlib
import uuid
import time

from pathlib import Path
from datetime import datetime, timezone
from typing import Optional

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Project Configuration

This section defines the project directory structure,
storage locations, database path, and pipeline configuration.

All project files are maintained inside the candidate_workspace
directory.

In [238]:
# Define project base directory

BASE_DIR = Path.cwd()

print("Project Base Directory:", BASE_DIR)

Project Base Directory: C:\Users\Administrator\CandidateWorkspace


## Define Data Zone Paths

In [239]:
# Define data directories

DATA_DIR = BASE_DIR / "data"

INCOMING_DIR = DATA_DIR / "incoming"
RAW_DIR = DATA_DIR / "raw"
VALIDATED_DIR = DATA_DIR / "validated"
CURATED_DIR = DATA_DIR / "curated"
QUARANTINE_DIR = DATA_DIR / "quarantine"

# Define supporting directories

WAREHOUSE_DIR = BASE_DIR / "warehouse"
LOG_DIR = BASE_DIR / "logs"
REPORT_DIR = BASE_DIR / "reports"
TEST_DIR = BASE_DIR / "tests"

print("All project paths defined successfully!")

All project paths defined successfully!


## Creating Project Folders

In [240]:
# Create all project directories

directories = [
    INCOMING_DIR,
    RAW_DIR,
    VALIDATED_DIR,
    CURATED_DIR,
    QUARANTINE_DIR,
    WAREHOUSE_DIR,
    LOG_DIR,
    REPORT_DIR,
    TEST_DIR
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print("All project folders created successfully!")

All project folders created successfully!


## Configure Database

In [241]:
# Define DuckDB database path

DB_PATH = WAREHOUSE_DIR / "warehouse.duckdb"

# Connect to DuckDB

conn = duckdb.connect(str(DB_PATH))

print("DuckDB connected successfully!")
print("Database Location:", DB_PATH)

ConnectionException: Connection Error: Can't open a connection to same database file with a different configuration than existing connections

## Configure Pipeline Settings

In [242]:
# Pipeline configuration

PIPELINE_NAME = "reliable_file_ingestion_pipeline"

RUN_ID = str(uuid.uuid4())

SOURCE_FILE_FORMAT = ".csv"

RANDOM_SEED = 42

EXPECTED_COLUMNS = [
    "order_id",
    "customer_id",
    "product_id",
    "order_date",
    "quantity",
    "unit_price",
    "order_status",
    "updated_at"
]

print("Pipeline Name:", PIPELINE_NAME)
print("Run ID:", RUN_ID)
print("Expected Columns:", EXPECTED_COLUMNS)

Pipeline Name: reliable_file_ingestion_pipeline
Run ID: a14fa797-33aa-49fb-87be-45c7110d5084
Expected Columns: ['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'unit_price', 'order_status', 'updated_at']


## Verify Project Structure

In [7]:
# Display project directory structure

for path in sorted(BASE_DIR.rglob("*")):
    if path.is_dir():
        print("📁", path.relative_to(BASE_DIR))

📁 .ipynb_checkpoints
📁 data
📁 data\curated
📁 data\incoming
📁 data\quarantine
📁 data\raw
📁 data\validated
📁 logs
📁 reports
📁 tests
📁 warehouse


## 3. Generate Sample Dataset

This section generates reproducible synthetic e-commerce
order data for testing the ingestion pipeline.

The dataset contains normal records and deliberately
introduced data-quality issues, including duplicates,
missing values, invalid quantities, invalid prices,
invalid reference keys, and late-arriving records.

A fixed random seed is used to ensure reproducibility.

In [8]:
# Define source dataset schema

source_schema = {
    "order_id": "integer",
    "customer_id": "integer",
    "product_id": "integer",
    "order_date": "datetime",
    "quantity": "integer",
    "unit_price": "float",
    "order_status": "string",
    "updated_at": "datetime"
}

schema_df = pd.DataFrame(
    list(source_schema.items()),
    columns=["column_name", "data_type"]
)

display(schema_df)

,column_name,data_type
0,order_id,integer
1,customer_id,integer
2,product_id,integer
3,order_date,datetime
4,quantity,integer
5,unit_price,float
6,order_status,string
7,updated_at,datetime


In [9]:
# Generate customer reference data

customers = pd.DataFrame({
    "customer_id": range(1, 201),
    "customer_name": [
        f"Customer_{i}" for i in range(1, 201)
    ]
})

# Generate product reference data

products = pd.DataFrame({
    "product_id": range(1, 51),
    "product_name": [
        f"Product_{i}" for i in range(1, 51)
    ]
})

print("Customers:", len(customers))
print("Products:", len(products))

Customers: 200
Products: 50


In [10]:
customers.to_csv(
    DATA_DIR / "customers.csv",
    index=False
)

products.to_csv(
    DATA_DIR / "products.csv",
    index=False
)

print("Reference datasets saved successfully!")

Reference datasets saved successfully!


In [11]:
def generate_orders(
    num_records=1000,
    seed=42
):

    rng = np.random.default_rng(seed)

    order_ids = np.arange(10001, 10001 + num_records)

    start_date = pd.Timestamp("2026-10-01")

    order_dates = (
        start_date
        + pd.to_timedelta(
            rng.integers(0, 45, num_records),
            unit="D"
        )
    )

    df = pd.DataFrame({

        "order_id": order_ids,

        "customer_id": rng.integers(
            1, 201, num_records
        ),

        "product_id": rng.integers(
            1, 51, num_records
        ),

        "order_date": order_dates,

        "quantity": rng.integers(
            1, 11, num_records
        ),

        "unit_price": np.round(
            rng.uniform(100, 5000, num_records), 2
        ),

        "order_status": rng.choice(
            ["Completed", "Pending", "Cancelled"],
            num_records
        ),

        "updated_at": order_dates + pd.Timedelta(hours=2)
    })

    return df

In [12]:
# Generate normal dataset

orders_df = generate_orders(
    num_records=1000,
    seed=RANDOM_SEED
)

print("Dataset generated successfully!")

print("Total records:", len(orders_df))

display(orders_df.head(10))

Dataset generated successfully!
Total records: 1000


,order_id,customer_id,product_id,order_date,quantity,unit_price,order_status,updated_at
0,10001,162,44,2026-10-05,8,4229.39,Pending,2026-10-05 02:00:00
1,10002,148,4,2026-11-04,7,2287.31,Cancelled,2026-11-04 02:00:00
2,10003,112,20,2026-10-30,10,4769.58,Pending,2026-10-30 02:00:00
3,10004,178,23,2026-10-20,10,3288.89,Pending,2026-10-20 02:00:00
4,10005,26,40,2026-10-20,8,667.90,Pending,2026-10-20 02:00:00
5,10006,185,7,2026-11-08,4,4436.96,Completed,2026-11-08 02:00:00
6,10007,171,22,2026-10-04,6,2497.40,Completed,2026-10-04 02:00:00
7,10008,101,8,2026-11-01,5,794.03,Cancelled,2026-11-01 02:00:00
8,10009,147,40,2026-10-10,7,848.18,Completed,2026-10-10 02:00:00
9,10010,105,32,2026-10-05,9,3468.92,Completed,2026-10-05 02:00:00


In [13]:
# Create a copy for introducing data-quality issues

orders_df = orders_df.copy()

# 1. Missing customer IDs
orders_df.loc[10:19, "customer_id"] = np.nan

# 2. Negative quantities
orders_df.loc[20:29, "quantity"] = -2

# 3. Invalid prices
orders_df.loc[30:39, "unit_price"] = -100

# 4. Invalid customer references
orders_df.loc[40:49, "customer_id"] = 9999

# 5. Invalid order status
orders_df.loc[50:59, "order_status"] = "Unknown"

# 6. Missing product IDs
orders_df.loc[60:69, "product_id"] = np.nan

# 7. Duplicate records
duplicate_records = orders_df.iloc[70:80].copy()

orders_df = pd.concat(
    [orders_df, duplicate_records],
    ignore_index=True
)

# 8. Late-arriving historical records
orders_df.loc[80:89, "order_date"] = (
    pd.Timestamp("2026-10-02")
)

print("Data-quality issues introduced!")
print("Total records:", len(orders_df))

Data-quality issues introduced!
Total records: 1010


In [14]:
# Split dataset into two incoming files

file_1 = orders_df.iloc[:500].copy()

file_2 = orders_df.iloc[500:].copy()

file_1.to_csv(
    INCOMING_DIR / "orders_batch_01.csv",
    index=False
)

file_2.to_csv(
    INCOMING_DIR / "orders_batch_02.csv",
    index=False
)

print("Incoming files created successfully!")

print("Batch 1 records:", len(file_1))
print("Batch 2 records:", len(file_2))

Incoming files created successfully!
Batch 1 records: 500
Batch 2 records: 510


In [15]:
# Generate 100,000 records

large_orders_df = generate_orders(
    num_records=100000,
    seed=RANDOM_SEED
)

large_orders_df.to_csv(
    INCOMING_DIR / "orders_large.csv",
    index=False
)

print("Large dataset generated successfully!")
print("Total records:", len(large_orders_df))

Large dataset generated successfully!
Total records: 100000


In [16]:
# List incoming files

incoming_files = list(INCOMING_DIR.glob("*.csv"))

for file in incoming_files:

    print(
        file.name,
        "| Size:",
        round(file.stat().st_size / 1024, 2),
        "KB"
    )

orders_batch_01.csv | Size: 32.97 KB
orders_batch_02.csv | Size: 33.67 KB
orders_large.csv | Size: 6204.31 KB


In [17]:
# Dataset overview

print("Dataset Shape:", orders_df.shape)

print("\nColumn Information:")
orders_df.info()

print("\nMissing Values:")
display(orders_df.isnull().sum())

print("\nDuplicate Order IDs:")
print(orders_df["order_id"].duplicated().sum())

print("\nSample Records:")
display(orders_df.head())

Dataset Shape: (1010, 8)

Column Information:
<class 'pandas.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      1010 non-null   int64         
 1   customer_id   1000 non-null   float64       
 2   product_id    1000 non-null   float64       
 3   order_date    1010 non-null   datetime64[us]
 4   quantity      1010 non-null   int64         
 5   unit_price    1010 non-null   float64       
 6   order_status  1010 non-null   str           
 7   updated_at    1010 non-null   datetime64[us]
dtypes: datetime64[us](2), float64(3), int64(2), str(1)
memory usage: 63.3 KB

Missing Values:


order_id         0
customer_id     10
product_id      10
order_date       0
quantity         0
unit_price       0
order_status     0
updated_at       0
dtype: int64


Duplicate Order IDs:
10

Sample Records:


,order_id,customer_id,product_id,order_date,quantity,unit_price,order_status,updated_at
0,10001,162.0,44.0,2026-10-05,8,4229.39,Pending,2026-10-05 02:00:00
1,10002,148.0,4.0,2026-11-04,7,2287.31,Cancelled,2026-11-04 02:00:00
2,10003,112.0,20.0,2026-10-30,10,4769.58,Pending,2026-10-30 02:00:00
3,10004,178.0,23.0,2026-10-20,10,3288.89,Pending,2026-10-20 02:00:00
4,10005,26.0,40.0,2026-10-20,8,667.90,Pending,2026-10-20 02:00:00


## 4. Data Ingestion

This section implements the ingestion layer of the data pipeline.

The ingestion process discovers incoming CSV files, reads source
data, generates file checksums, captures ingestion metadata,
and stores original files in the Raw Zone.

The original source data is preserved without modification
to support traceability, auditing, replay, and recovery.

In [18]:
# Define ingestion metadata structure

ingestion_metadata = []

print("Ingestion metadata initialized!")

Ingestion metadata initialized!


In [19]:
def discover_files():

    """
    Discover CSV files available in incoming directory.
    """

    files = sorted(INCOMING_DIR.glob("*.csv"))

    print(f"Total files discovered: {len(files)}")

    for file in files:
        print("Discovered:", file.name)

    return files

In [20]:
incoming_files = discover_files()

Total files discovered: 3
Discovered: orders_batch_01.csv
Discovered: orders_batch_02.csv
Discovered: orders_large.csv


In [21]:
def generate_file_checksum(file_path):

    """
    Generate SHA-256 checksum for a source file.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:

        for chunk in iter(
            lambda: file.read(4096),
            b""
        ):
            sha256.update(chunk)

    return sha256.hexdigest()

In [22]:
sample_file = incoming_files[0]

checksum = generate_file_checksum(sample_file)

print("File:", sample_file.name)
print("SHA-256:", checksum)

File: orders_batch_01.csv
SHA-256: d52adea983a0c02a2c9264282edbac41ce5a80c7537fd2533c55632e266fb5d8


In [23]:
def read_source_file(file_path):

    """
    Read source CSV file using Pandas.
    """

    try:

        df = pd.read_csv(file_path)

        print(f"Successfully read: {file_path.name}")
        print("Records:", len(df))

        return df

    except Exception as error:

        print(f"Failed to read {file_path.name}: {error}")

        raise

In [24]:
sample_df = read_source_file(incoming_files[0])

display(sample_df.head())

Successfully read: orders_batch_01.csv
Records: 500


,order_id,customer_id,product_id,order_date,quantity,unit_price,order_status,updated_at
0,10001,162.0,44.0,2026-10-05,8,4229.39,Pending,2026-10-05 02:00:00
1,10002,148.0,4.0,2026-11-04,7,2287.31,Cancelled,2026-11-04 02:00:00
2,10003,112.0,20.0,2026-10-30,10,4769.58,Pending,2026-10-30 02:00:00
3,10004,178.0,23.0,2026-10-20,10,3288.89,Pending,2026-10-20 02:00:00
4,10005,26.0,40.0,2026-10-20,8,667.90,Pending,2026-10-20 02:00:00


## Implement raw file storage

In [25]:
import shutil

def copy_to_raw(file_path):

    """
    Copy original source file into Raw Zone.
    """

    destination = RAW_DIR / file_path.name

    if not destination.exists():

        shutil.copy2(file_path, destination)

        print(f"File copied to Raw Zone: {destination.name}")

    else:

        print(f"Raw file already exists: {destination.name}")

    return destination

In [26]:
raw_file = copy_to_raw(incoming_files[0])

print("Raw file location:", raw_file)

File copied to Raw Zone: orders_batch_01.csv
Raw file location: C:\Users\Administrator\CandidateWorkspace\data\raw\orders_batch_01.csv


## Implement ingestion metadata tracking

In [27]:
def register_ingestion_metadata(file_path, df, checksum):

    """
    Register source file ingestion details.
    """

    metadata = {
        "file_name": file_path.name,
        "file_checksum": checksum,
        "file_size": file_path.stat().st_size,
        "ingestion_time": datetime.now(timezone.utc).isoformat(),
        "run_id": RUN_ID,
        "file_status": "INGESTED",
        "record_count": len(df)
    }

    return metadata

In [28]:
sample_metadata = register_ingestion_metadata(
    sample_file,
    sample_df,
    checksum
)

display(pd.DataFrame([sample_metadata]))

,file_name,file_checksum,file_size,ingestion_time,run_id,file_status,record_count
0,orders_batch_01.csv,d52adea983a0c02a2c9264282edbac41ce5a80c7537fd2...,33765,2026-11-14T09:08:54.453722+00:00,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,INGESTED,500


## Implement complete ingestion process

In [29]:
def ingest_files():

    """
    Discover, read, checksum and store incoming files
    in the Raw Zone.
    """

    metadata_records = []

    files = discover_files()

    for file_path in files:

        try:

            # Generate file checksum
            checksum = generate_file_checksum(file_path)

            # Read source file
            df = read_source_file(file_path)

            # Preserve original file in Raw Zone
            copy_to_raw(file_path)

            # Register metadata
            metadata = register_ingestion_metadata(
                file_path,
                df,
                checksum
            )

            metadata_records.append(metadata)

            print(f"Ingestion completed: {file_path.name}")

        except Exception as error:

            print(f"Ingestion failed: {file_path.name}")
            print("Error:", error)

    return pd.DataFrame(metadata_records)

In [30]:
ingestion_df = ingest_files()

display(ingestion_df)

Total files discovered: 3
Discovered: orders_batch_01.csv
Discovered: orders_batch_02.csv
Discovered: orders_large.csv
Successfully read: orders_batch_01.csv
Records: 500
Raw file already exists: orders_batch_01.csv
Ingestion completed: orders_batch_01.csv
Successfully read: orders_batch_02.csv
Records: 510
File copied to Raw Zone: orders_batch_02.csv
Ingestion completed: orders_batch_02.csv
Successfully read: orders_large.csv
Records: 100000
File copied to Raw Zone: orders_large.csv
Ingestion completed: orders_large.csv


,file_name,file_checksum,file_size,ingestion_time,run_id,file_status,record_count
0,orders_batch_01.csv,d52adea983a0c02a2c9264282edbac41ce5a80c7537fd2...,33765,2026-11-14T09:09:30.792477+00:00,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,INGESTED,500
1,orders_batch_02.csv,9daa655ea816c59c831d6067e1131a619e2a95680760fd...,34481,2026-11-14T09:09:30.836299+00:00,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,INGESTED,510
2,orders_large.csv,f9829be26ed28682bb6b239635b9b1922ef6019ff76391...,6353216,2026-11-14T09:09:31.350254+00:00,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,INGESTED,100000


## Save ingestion metadata

In [31]:
# Save ingestion metadata

metadata_file = REPORT_DIR / "ingestion_metadata.csv"

ingestion_df.to_csv(
    metadata_file,
    index=False
)

print("Ingestion metadata saved successfully!")
print("Location:", metadata_file)

Ingestion metadata saved successfully!
Location: C:\Users\Administrator\CandidateWorkspace\reports\ingestion_metadata.csv


In [32]:
# Verify files stored in Raw Zone

raw_files = sorted(RAW_DIR.glob("*.csv"))

print("Total files in Raw Zone:", len(raw_files))

for file in raw_files:

    print(
        file.name,
        "| Size:",
        round(file.stat().st_size / 1024, 2),
        "KB"
    )

Total files in Raw Zone: 3
orders_batch_01.csv | Size: 32.97 KB
orders_batch_02.csv | Size: 33.67 KB
orders_large.csv | Size: 6204.31 KB


In [33]:
print("========== INGESTION SUMMARY ==========")

print("Pipeline Name:", PIPELINE_NAME)
print("Run ID:", RUN_ID)

print("Files Discovered:", len(incoming_files))
print("Files Ingested:", len(ingestion_df))

print(
    "Total Input Records:",
    ingestion_df["record_count"].sum()
)

print(
    "Total Raw Files:",
    len(list(RAW_DIR.glob("*.csv")))
)

print("=======================================")

========== INGESTION SUMMARY ==========
Pipeline Name: reliable_file_ingestion_pipeline
Run ID: 3baf8b71-75e0-45fe-bd0e-1c1dc728212a
Files Discovered: 3
Files Ingested: 3
Total Input Records: 101010
Total Raw Files: 3


## 5. Schema Validation and Data Quality Checks

This section validates ingested source records against
predefined schema requirements and business rules.

The validation engine checks:

- Required columns
- Missing mandatory values
- Data types
- Duplicate order IDs
- Invalid quantities and prices
- Invalid order statuses
- Customer and product referential integrity
- Invalid timestamps

Valid records are separated from rejected records.
Each rejected record is stored with its source filename,
run ID, and rejection reason.

## Define validation rules

In [34]:
# Define mandatory columns

REQUIRED_COLUMNS = [
    "order_id",
    "customer_id",
    "product_id",
    "order_date",
    "quantity",
    "unit_price",
    "order_status",
    "updated_at"
]

# Define allowed order statuses

VALID_STATUSES = [
    "Completed",
    "Pending",
    "Cancelled"
]

# Define business rules

MIN_QUANTITY = 1

MIN_UNIT_PRICE = 0

print("Validation rules configured successfully!")

Validation rules configured successfully!


## Read raw source files

In [35]:
# Read all raw CSV files

raw_dataframes = []

for file_path in sorted(RAW_DIR.glob("*.csv")):

    df = pd.read_csv(file_path)

    df["source_file"] = file_path.name

    raw_dataframes.append(df)

    print(file_path.name, ":", len(df), "records")

raw_df = pd.concat(
    raw_dataframes,
    ignore_index=True
)

print("\nTotal Raw Records:", len(raw_df))

display(raw_df.head())

orders_batch_01.csv : 500 records
orders_batch_02.csv : 510 records
orders_large.csv : 100000 records

Total Raw Records: 101010


,order_id,customer_id,product_id,order_date,quantity,unit_price,order_status,updated_at,source_file
0,10001,162.0,44.0,2026-10-05,8,4229.39,Pending,2026-10-05 02:00:00,orders_batch_01.csv
1,10002,148.0,4.0,2026-11-04,7,2287.31,Cancelled,2026-11-04 02:00:00,orders_batch_01.csv
2,10003,112.0,20.0,2026-10-30,10,4769.58,Pending,2026-10-30 02:00:00,orders_batch_01.csv
3,10004,178.0,23.0,2026-10-20,10,3288.89,Pending,2026-10-20 02:00:00,orders_batch_01.csv
4,10005,26.0,40.0,2026-10-20,8,667.90,Pending,2026-10-20 02:00:00,orders_batch_01.csv


## Validate schema

In [36]:
def validate_schema(df):

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Schema validation failed. Missing columns: "
            f"{missing_columns}"
        )

    print("Schema validation successful!")

    return True

In [37]:
validate_schema(raw_df)

Schema validation successful!


True

## Validate data types

In [38]:
def validate_data_types(df):

    df = df.copy()

    df["order_id"] = pd.to_numeric(
        df["order_id"],
        errors="coerce"
    )

    df["customer_id"] = pd.to_numeric(
        df["customer_id"],
        errors="coerce"
    )

    df["product_id"] = pd.to_numeric(
        df["product_id"],
        errors="coerce"
    )

    df["quantity"] = pd.to_numeric(
        df["quantity"],
        errors="coerce"
    )

    df["unit_price"] = pd.to_numeric(
        df["unit_price"],
        errors="coerce"
    )

    df["order_date"] = pd.to_datetime(
        df["order_date"],
        errors="coerce"
    )

    df["updated_at"] = pd.to_datetime(
        df["updated_at"],
        errors="coerce"
    )

    return df

In [39]:
typed_df = validate_data_types(raw_df)

print("Data type conversion completed!")

display(typed_df.dtypes)

Data type conversion completed!


order_id                 int64
customer_id            float64
product_id             float64
order_date      datetime64[us]
quantity                 int64
unit_price             float64
order_status               str
updated_at      datetime64[us]
source_file                str
dtype: object

## Validate missing values

In [40]:
def validate_nulls(df):

    null_rules = {
        "order_id": "Missing order ID",
        "customer_id": "Missing customer ID",
        "product_id": "Missing product ID",
        "order_date": "Missing or invalid order date",
        "quantity": "Missing or invalid quantity",
        "unit_price": "Missing or invalid unit price",
        "order_status": "Missing order status",
        "updated_at": "Missing or invalid update timestamp"
    }

    errors = pd.DataFrame(
        "",
        index=df.index,
        columns=["validation_errors"]
    )

    for column, reason in null_rules.items():

        mask = df[column].isna()

        errors.loc[mask, "validation_errors"] += reason + "; "

    return errors["validation_errors"]

In [41]:
null_errors = validate_nulls(typed_df)

print("Records with missing required values:",
      (null_errors != "").sum())

Records with missing required values: 20


## Validate business rules

In [42]:
def validate_business_rules(df):

    errors = pd.Series(
        "",
        index=df.index,
        dtype="object"
    )

    # Quantity must be positive
    errors.loc[
        df["quantity"].notna() &
        (df["quantity"] <= 0)
    ] += "Invalid quantity; "

    # Quantity must be an integer
    errors.loc[
        df["quantity"].notna() &
        (df["quantity"] % 1 != 0)
    ] += "Quantity must be an integer; "

    # Price must be positive
    errors.loc[
        df["unit_price"].notna() &
        (df["unit_price"] <= 0)
    ] += "Invalid unit price; "

    # Order ID must be positive integer
    errors.loc[
        df["order_id"].notna() &
        (
            (df["order_id"] <= 0) |
            (df["order_id"] % 1 != 0)
        )
    ] += "Invalid order ID; "

    # Customer ID must be a positive integer
    errors.loc[
        df["customer_id"].notna() &
        (
            (df["customer_id"] <= 0) |
            (df["customer_id"] % 1 != 0)
        )
    ] += "Invalid customer ID; "

    # Product ID must be a positive integer
    errors.loc[
        df["product_id"].notna() &
        (
            (df["product_id"] <= 0) |
            (df["product_id"] % 1 != 0)
        )
    ] += "Invalid product ID; "

    # Status must be valid
    errors.loc[
        df["order_status"].notna() &
        (~df["order_status"].isin(VALID_STATUSES))
    ] += "Invalid order status; "

    # Order date cannot be in the future
    errors.loc[
        df["order_date"].notna() &
        (df["order_date"] > pd.Timestamp.now())
    ] += "Future order date; "

    # Updated timestamp cannot precede order date
    errors.loc[
        df["updated_at"].notna() &
        df["order_date"].notna() &
        (df["updated_at"] < df["order_date"])
    ] += "Update timestamp before order date; "

    return errors

In [43]:
business_errors = validate_business_rules(typed_df)

print("Business rule violations:",
      (business_errors != "").sum())

Business rule violations: 30


## Validate duplicate order IDs

In [44]:
def validate_duplicates(df):

    duplicate_mask = df.duplicated(
        subset=["order_id"],
        keep="first"
    )

    errors = pd.Series(
        "",
        index=df.index,
        dtype="object"
    )

    errors.loc[duplicate_mask] = "Duplicate order ID; "

    return errors

In [45]:
duplicate_errors = validate_duplicates(typed_df)

print(
    "Duplicate records detected:",
    (duplicate_errors != "").sum()
)

Duplicate records detected: 1010


## Validate customer and product references

In [46]:
def validate_foreign_keys(df):

    errors = pd.Series(
        "",
        index=df.index,
        dtype="object"
    )

    valid_customer_ids = set(
        customers["customer_id"]
    )

    valid_product_ids = set(
        products["product_id"]
    )

    invalid_customers = (
        df["customer_id"].notna() &
        ~df["customer_id"].isin(valid_customer_ids)
    )

    invalid_products = (
        df["product_id"].notna() &
        ~df["product_id"].isin(valid_product_ids)
    )

    errors.loc[invalid_customers] += (
        "Invalid customer reference; "
    )

    errors.loc[invalid_products] += (
        "Invalid product reference; "
    )

    return errors

In [47]:
foreign_key_errors = validate_foreign_keys(typed_df)

print(
    "Invalid reference records:",
    (foreign_key_errors != "").sum()
)

Invalid reference records: 10


## Combine all validation results

In [48]:
def validate_records(df):

    df = df.copy()

    # Individual validation checks
    null_errors = validate_nulls(df)

    business_errors = validate_business_rules(df)

    duplicate_errors = validate_duplicates(df)

    foreign_key_errors = validate_foreign_keys(df)

    # Combine validation errors
    df["rejection_reason"] = (
        null_errors +
        business_errors +
        duplicate_errors +
        foreign_key_errors
    )

    # Remove empty error separators
    df["rejection_reason"] = (
        df["rejection_reason"]
        .str.strip()
        .str.strip(";")
    )

    # Add execution metadata
    df["run_id"] = RUN_ID

    df["validation_timestamp"] = (
        datetime.now(timezone.utc).isoformat()
    )

    # Separate valid and invalid records
    valid_records = df[
        df["rejection_reason"] == ""
    ].copy()

    rejected_records = df[
        df["rejection_reason"] != ""
    ].copy()

    return valid_records, rejected_records

In [49]:
valid_records, rejected_records = validate_records(
    typed_df
)

print("Validation completed!")

print("Total Records:", len(typed_df))

print("Valid Records:", len(valid_records))

print("Rejected Records:", len(rejected_records))

Validation completed!
Total Records: 101010
Valid Records: 99940
Rejected Records: 1070


## Inspect rejected records

In [50]:
# Display rejected records

display(
    rejected_records[
        [
            "order_id",
            "customer_id",
            "product_id",
            "quantity",
            "unit_price",
            "source_file",
            "rejection_reason"
        ]
    ].head(20)
)

,order_id,customer_id,product_id,quantity,unit_price,source_file,rejection_reason
10,10011,NaN,25.0,4,332.13,orders_batch_01.csv,Missing customer ID
11,10012,NaN,20.0,6,1504.31,orders_batch_01.csv,Missing customer ID
12,10013,NaN,8.0,1,784.96,orders_batch_01.csv,Missing customer ID
13,10014,NaN,47.0,8,2261.28,orders_batch_01.csv,Missing customer ID
14,10015,NaN,33.0,5,2130.33,orders_batch_01.csv,Missing customer ID
15,10016,NaN,16.0,5,2657.19,orders_batch_01.csv,Missing customer ID
16,10017,NaN,21.0,6,1470.22,orders_batch_01.csv,Missing customer ID
17,10018,NaN,37.0,3,257.01,orders_batch_01.csv,Missing customer ID
18,10019,NaN,36.0,6,3110.49,orders_batch_01.csv,Missing customer ID
19,10020,NaN,24.0,8,1931.22,orders_batch_01.csv,Missing customer ID


## Save Validated and Quarantine records

In [51]:
# Save valid records

validated_file = VALIDATED_DIR / "validated_orders.csv"

valid_records.to_csv(
    validated_file,
    index=False
)

# Save rejected records

quarantine_file = (
    QUARANTINE_DIR / "rejected_orders.csv"
)

rejected_records.to_csv(
    quarantine_file,
    index=False
)

print("Validated records saved:", validated_file)

print("Rejected records saved:", quarantine_file)

Validated records saved: C:\Users\Administrator\CandidateWorkspace\data\validated\validated_orders.csv
Rejected records saved: C:\Users\Administrator\CandidateWorkspace\data\quarantine\rejected_orders.csv


## Generate data-quality report

In [52]:
# Generate data-quality summary

quality_report = pd.DataFrame({

    "metric": [
        "Total Input Records",
        "Valid Records",
        "Rejected Records",
        "Duplicate Records",
        "Missing Value Violations",
        "Business Rule Violations",
        "Foreign Key Violations"
    ],

    "count": [
        len(typed_df),
        len(valid_records),
        len(rejected_records),
        (duplicate_errors != "").sum(),
        (null_errors != "").sum(),
        (business_errors != "").sum(),
        (foreign_key_errors != "").sum()
    ]

})

display(quality_report)

,metric,count
0,Total Input Records,101010
1,Valid Records,99940
2,Rejected Records,1070
3,Duplicate Records,1010
4,Missing Value Violations,20
5,Business Rule Violations,30
6,Foreign Key Violations,10


In [53]:
quality_report.to_csv(
    REPORT_DIR / "quality_report.csv",
    index=False
)

print("Data quality report saved successfully!")

Data quality report saved successfully!


## Verify output folders

In [54]:
print("========== DATA QUALITY SUMMARY ==========")

print("Input Records:", len(typed_df))

print("Valid Records:", len(valid_records))

print("Rejected Records:", len(rejected_records))

print(
    "Reconciliation:",
    len(valid_records) + len(rejected_records)
)

print("\nValidated Folder Files:")

for file in VALIDATED_DIR.glob("*"):
    print(file.name)

print("\nQuarantine Folder Files:")

for file in QUARANTINE_DIR.glob("*"):
    print(file.name)

print("==========================================")

========== DATA QUALITY SUMMARY ==========
Input Records: 101010
Valid Records: 99940
Rejected Records: 1070
Reconciliation: 101010

Validated Folder Files:
validated_orders.csv

Quarantine Folder Files:
rejected_orders.csv


In [55]:
raw_dataframes = []

for file_path in sorted(
    RAW_DIR.glob("orders_batch_*.csv")
):

    df = pd.read_csv(file_path)

    df["source_file"] = file_path.name

    raw_dataframes.append(df)

raw_df = pd.concat(
    raw_dataframes,
    ignore_index=True
)

print("Functional input records:", len(raw_df))

Functional input records: 1010


## 6. Duplicate Handling and Idempotency

This section implements file-level and record-level
duplicate detection.

File-level duplicates are identified using SHA-256 checksums.

Record-level duplicates are identified using business keys
such as order_id.

The pipeline maintains persistent ingestion metadata
to prevent repeated processing of previously ingested files.

The deduplication process is deterministic and supports
safe reruns and recovery.

## Create persistent file registry in DuckDB

In [56]:
# Create persistent file ingestion registry

conn.execute("""
CREATE TABLE IF NOT EXISTS file_ingestion_registry (

    file_checksum VARCHAR PRIMARY KEY,

    file_name VARCHAR,

    file_size BIGINT,

    ingestion_time VARCHAR,

    run_id VARCHAR,

    file_status VARCHAR,

    record_count BIGINT

)
""")

print("File ingestion registry created successfully!")

File ingestion registry created successfully!


In [57]:
# Load existing ingestion metadata

metadata_df = pd.read_csv(
    REPORT_DIR / "ingestion_metadata.csv"
)

display(metadata_df.head())

,file_name,file_checksum,file_size,ingestion_time,run_id,file_status,record_count
0,orders_batch_01.csv,d52adea983a0c02a2c9264282edbac41ce5a80c7537fd2...,33765,2026-11-14T09:09:30.792477+00:00,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,INGESTED,500
1,orders_batch_02.csv,9daa655ea816c59c831d6067e1131a619e2a95680760fd...,34481,2026-11-14T09:09:30.836299+00:00,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,INGESTED,510
2,orders_large.csv,f9829be26ed28682bb6b239635b9b1922ef6019ff76391...,6353216,2026-11-14T09:09:31.350254+00:00,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,INGESTED,100000


In [58]:
# Register previously ingested files

for _, row in metadata_df.iterrows():

    conn.execute("""
        INSERT INTO file_ingestion_registry
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT DO NOTHING
    """, [
        row["file_checksum"],
        row["file_name"],
        int(row["file_size"]),
        str(row["ingestion_time"]),
        row["run_id"],
        row["file_status"],
        int(row["record_count"])
    ])

print("Existing file metadata registered successfully!")

Existing file metadata registered successfully!


## Implement file-level duplicate detectio

In [59]:
def is_duplicate_file(file_path):

    """
    Check whether a file was previously ingested
    using its SHA-256 checksum.
    """

    checksum = generate_file_checksum(file_path)

    result = conn.execute("""
        SELECT COUNT(*)
        FROM file_ingestion_registry
        WHERE file_checksum = ?
    """, [checksum]).fetchone()

    return result[0] > 0

In [60]:
# Check an already ingested file

test_file = RAW_DIR / "orders_batch_01.csv"

duplicate_status = is_duplicate_file(test_file)

print("File:", test_file.name)

print("Already Ingested:", duplicate_status)

File: orders_batch_01.csv
Already Ingested: True


## Implement safe file ingestion

In [61]:
def safe_file_ingestion(file_path):

    """
    Ingest only new files.
    Skip previously ingested files.
    """

    checksum = generate_file_checksum(file_path)

    if is_duplicate_file(file_path):

        print(f"Duplicate file detected: {file_path.name}")

        return "SKIPPED_DUPLICATE"

    try:

        df = read_source_file(file_path)

        raw_destination = copy_to_raw(file_path)

        metadata = register_ingestion_metadata(
            file_path,
            df,
            checksum
        )

        conn.execute("""
            INSERT INTO file_ingestion_registry
            VALUES (?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT DO NOTHING
        """, [
            metadata["file_checksum"],
            metadata["file_name"],
            metadata["file_size"],
            metadata["ingestion_time"],
            metadata["run_id"],
            "INGESTED",
            metadata["record_count"]
        ])

        print(f"Successfully ingested: {file_path.name}")

        return "INGESTED"

    except Exception as error:

        print("File ingestion failed:", file_path.name)

        print("Error:", error)

        return "FAILED"

In [62]:
# Simulate duplicate file delivery

original_file = RAW_DIR / "orders_batch_01.csv"

duplicate_test_file = (
    INCOMING_DIR / "orders_batch_01_copy.csv"
)

shutil.copy2(
    original_file,
    duplicate_test_file
)

print("Duplicate test file created!")

Duplicate test file created!


In [63]:
# Test duplicate detection

result = safe_file_ingestion(duplicate_test_file)

print("Processing Result:", result)

Duplicate file detected: orders_batch_01_copy.csv
Processing Result: SKIPPED_DUPLICATE


## Implement record-level deduplication

In [64]:
def deduplicate_records(df):

    """
    Remove duplicate order IDs while retaining
    the first occurrence.
    """

    df = df.copy()

    duplicate_mask = df.duplicated(
        subset=["order_id"],
        keep="first"
    )

    duplicate_records = df.loc[
        duplicate_mask
    ].copy()

    deduplicated_records = df.loc[
        ~duplicate_mask
    ].copy()

    duplicate_records["rejection_reason"] = (
        "Duplicate order ID"
    )

    return deduplicated_records, duplicate_records

In [65]:
# Keep records that passed all checks except duplicate detection

non_duplicate_error_mask = (
    (null_errors == "") &
    (business_errors == "") &
    (foreign_key_errors == "")
)

dedup_candidates = typed_df.loc[
    non_duplicate_error_mask
].copy()

print("Records eligible for deduplication:",
      len(dedup_candidates))

Records eligible for deduplication: 100950


In [66]:
deduplicated_records, duplicate_record_rows = (
    deduplicate_records(dedup_candidates)
)

print("Records after deduplication:",
      len(deduplicated_records))

print("Duplicate records identified:",
      len(duplicate_record_rows))

Records after deduplication: 100000
Duplicate records identified: 950


## Save deduplicated outputs

In [67]:
# Save deduplicated records

deduplicated_file = (
    VALIDATED_DIR / "deduplicated_orders.csv"
)

deduplicated_records.to_csv(
    deduplicated_file,
    index=False
)

# Save duplicate records separately for auditing

duplicate_file = (
    QUARANTINE_DIR / "duplicate_orders.csv"
)

duplicate_record_rows.to_csv(
    duplicate_file,
    index=False
)

print("Deduplicated data saved successfully!")
print("Duplicate audit file saved successfully!")

Deduplicated data saved successfully!
Duplicate audit file saved successfully!


## Verify idempotency through repeated execution

In [68]:
# First execution

first_run = safe_file_ingestion(
    RAW_DIR / "orders_batch_01.csv"
)

print("First execution:", first_run)

Duplicate file detected: orders_batch_01.csv
First execution: SKIPPED_DUPLICATE


In [69]:
# Second execution

second_run = safe_file_ingestion(
    RAW_DIR / "orders_batch_01.csv"
)

print("Second execution:", second_run)

Duplicate file detected: orders_batch_01.csv
Second execution: SKIPPED_DUPLICATE


In [70]:
# View file ingestion registry

registry_df = conn.execute("""
    SELECT
        file_name,
        file_checksum,
        file_status,
        record_count
    FROM file_ingestion_registry
    ORDER BY file_name
""").df()

display(registry_df)

,file_name,file_checksum,file_status,record_count
0,orders_batch_01.csv,d52adea983a0c02a2c9264282edbac41ce5a80c7537fd2...,INGESTED,500
1,orders_batch_02.csv,9daa655ea816c59c831d6067e1131a619e2a95680760fd...,INGESTED,510
2,orders_large.csv,f9829be26ed28682bb6b239635b9b1922ef6019ff76391...,INGESTED,100000


In [71]:
registry_count = conn.execute("""
    SELECT COUNT(*)
    FROM file_ingestion_registry
""").fetchone()[0]

print("Registered unique files:", registry_count)

Registered unique files: 3


## Clean up the duplicate test file

In [72]:
if duplicate_test_file.exists():

    duplicate_test_file.unlink()

    print("Temporary duplicate test file removed!")

Temporary duplicate test file removed!


## Final duplicate handling report

In [73]:
print("========== DUPLICATE HANDLING SUMMARY ==========")

print(
    "Registered Files:",
    registry_count
)

print(
    "Records Eligible for Deduplication:",
    len(dedup_candidates)
)

print(
    "Duplicate Records:",
    len(duplicate_record_rows)
)

print(
    "Unique Records:",
    len(deduplicated_records)
)

print(
    "Duplicate Order IDs Remaining:",
    deduplicated_records["order_id"].duplicated().sum()
)

print("================================================")

========== DUPLICATE HANDLING SUMMARY ==========
Registered Files: 3
Records Eligible for Deduplication: 100950
Duplicate Records: 950
Unique Records: 100000
Duplicate Order IDs Remaining: 0


## 7. Data Transformation and Curated Zone

This section transforms validated and deduplicated records
into a clean analytical dataset.

The transformation process includes:

- Column standardization
- Data type standardization
- Timestamp normalization
- Derived column creation
- Total amount calculation
- Date-based partitioning
- Curated data publishing

The final transformed records are stored in the Curated Zone
for downstream analytical consumption.

## Load deduplicated records

In [74]:
# Load deduplicated records from Step 6

transformation_df = deduplicated_records.copy()

print("Records available for transformation:",
      len(transformation_df))

display(transformation_df.head())

Records available for transformation: 100000


,order_id,customer_id,product_id,order_date,quantity,unit_price,order_status,updated_at,source_file
0,10001,162.0,44.0,2026-10-05,8,4229.39,Pending,2026-10-05 02:00:00,orders_batch_01.csv
1,10002,148.0,4.0,2026-11-04,7,2287.31,Cancelled,2026-11-04 02:00:00,orders_batch_01.csv
2,10003,112.0,20.0,2026-10-30,10,4769.58,Pending,2026-10-30 02:00:00,orders_batch_01.csv
3,10004,178.0,23.0,2026-10-20,10,3288.89,Pending,2026-10-20 02:00:00,orders_batch_01.csv
4,10005,26.0,40.0,2026-10-20,8,667.90,Pending,2026-10-20 02:00:00,orders_batch_01.csv


## Standardize column names

In [75]:
def standardize_columns(df):

    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    return df

In [76]:
transformation_df = standardize_columns(
    transformation_df
)

print("Column names standardized!")

print(transformation_df.columns.tolist())

Column names standardized!
['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'unit_price', 'order_status', 'updated_at', 'source_file']


## Standardize data types

In [77]:
def standardize_data_types(df):

    df = df.copy()

    integer_columns = [
        "order_id",
        "customer_id",
        "product_id",
        "quantity"
    ]

    for column in integer_columns:

        df[column] = df[column].astype("int64")

    df["unit_price"] = df["unit_price"].astype("float64")

    df["order_date"] = pd.to_datetime(
        df["order_date"],
        utc=True
    )

    df["updated_at"] = pd.to_datetime(
        df["updated_at"],
        utc=True
    )

    df["order_status"] = (
        df["order_status"].astype("string").str.strip()
    )

    return df

In [78]:
transformation_df = standardize_data_types(
    transformation_df
)

display(transformation_df.dtypes)

order_id                      int64
customer_id                   int64
product_id                    int64
order_date      datetime64[us, UTC]
quantity                      int64
unit_price                  float64
order_status                 string
updated_at      datetime64[us, UTC]
source_file                     str
dtype: object

## Create derived business columns

In [79]:
def create_derived_columns(df):

    df = df.copy()

    # Calculate order amount
    df["total_amount"] = (
        df["quantity"] * df["unit_price"]
    )

    # Extract date components
    df["order_year"] = df["order_date"].dt.year

    df["order_month"] = df["order_date"].dt.month

    df["order_day"] = df["order_date"].dt.day

    # Create year-month partition key
    df["order_year_month"] = (
        df["order_date"].dt.strftime("%Y-%m")
    )

    return df

In [80]:
transformation_df = create_derived_columns(
    transformation_df
)

display(
    transformation_df[
        [
            "order_id",
            "quantity",
            "unit_price",
            "total_amount",
            "order_year",
            "order_month",
            "order_day"
        ]
    ].head(10)
)

,order_id,quantity,unit_price,total_amount,order_year,order_month,order_day
0,10001,8,4229.39,33835.12,2026,10,5
1,10002,7,2287.31,16011.17,2026,11,4
2,10003,10,4769.58,47695.80,2026,10,30
3,10004,10,3288.89,32888.90,2026,10,20
4,10005,8,667.90,5343.20,2026,10,20
5,10006,4,4436.96,17747.84,2026,11,8
6,10007,6,2497.40,14984.40,2026,10,4
7,10008,5,794.03,3970.15,2026,11,1
8,10009,7,848.18,5937.26,2026,10,10
9,10010,9,3468.92,31220.28,2026,10,5


## Add processing metadata

In [81]:
def add_processing_metadata(df):

    df = df.copy()

    df["run_id"] = RUN_ID

    df["processing_timestamp"] = (
        datetime.now(timezone.utc).isoformat()
    )

    return df

In [82]:
transformation_df = add_processing_metadata(
    transformation_df
)

print("Processing metadata added!")

Processing metadata added!


## Prepare final curated dataset

In [83]:
CURATED_COLUMNS = [
    "order_id",
    "customer_id",
    "product_id",
    "order_date",
    "quantity",
    "unit_price",
    "total_amount",
    "order_status",
    "updated_at",
    "order_year",
    "order_month",
    "order_day",
    "order_year_month",
    "source_file",
    "run_id",
    "processing_timestamp"
]

curated_df = transformation_df[
    CURATED_COLUMNS
].copy()

print("Curated dataset prepared successfully!")

print("Total curated records:", len(curated_df))

display(curated_df.head())

Curated dataset prepared successfully!
Total curated records: 100000


,order_id,customer_id,product_id,order_date,quantity,unit_price,total_amount,order_status,updated_at,order_year,order_month,order_day,order_year_month,source_file,run_id,processing_timestamp
0,10001,162,44,2026-10-05 00:00:00+00:00,8,4229.39,33835.12,Pending,2026-10-05 02:00:00+00:00,2026,10,5,2026-10,orders_batch_01.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:33:50.773142+00:00
1,10002,148,4,2026-11-04 00:00:00+00:00,7,2287.31,16011.17,Cancelled,2026-11-04 02:00:00+00:00,2026,11,4,2026-11,orders_batch_01.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:33:50.773142+00:00
2,10003,112,20,2026-10-30 00:00:00+00:00,10,4769.58,47695.80,Pending,2026-10-30 02:00:00+00:00,2026,10,30,2026-10,orders_batch_01.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:33:50.773142+00:00
3,10004,178,23,2026-10-20 00:00:00+00:00,10,3288.89,32888.90,Pending,2026-10-20 02:00:00+00:00,2026,10,20,2026-10,orders_batch_01.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:33:50.773142+00:00
4,10005,26,40,2026-10-20 00:00:00+00:00,8,667.90,5343.20,Pending,2026-10-20 02:00:00+00:00,2026,10,20,2026-10,orders_batch_01.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:33:50.773142+00:00


## Implement date-based partitioning

In [ ]:
## def save_partitioned_data(df):

    """
    Save curated data into year and month partitions.
    Partitions are based on order_date, not arrival date.
    """

    for (year, month), group in df.groupby(
        ["order_year", "order_month"]
    ):

        partition_dir = (
            CURATED_DIR
            / f"order_year={year}"
            / f"order_month={month:02d}"
        )

        partition_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        output_file = partition_dir / "orders.csv"

        group.to_csv(
            output_file,
            index=False
        )

        print(
            f"Saved {len(group)} records:",
            output_file
        )

    print("Date partitioning completed!")

In [85]:
save_partitioned_data(curated_df)

Saved 68886 records: C:\Users\Administrator\CandidateWorkspace\data\curated\order_year=2026\order_month=10\orders.csv
Saved 31114 records: C:\Users\Administrator\CandidateWorkspace\data\curated\order_year=2026\order_month=11\orders.csv
Date partitioning completed!


## Save complete curated dataset

In [86]:
curated_output_file = (
    CURATED_DIR / "curated_orders.csv"
)

curated_df.to_csv(
    curated_output_file,
    index=False
)

print("Curated dataset saved successfully!")

print("Location:", curated_output_file)

Curated dataset saved successfully!
Location: C:\Users\Administrator\CandidateWorkspace\data\curated\curated_orders.csv


In [87]:
print("========== CURATED DATA SUMMARY ==========")

print("Total Records:", len(curated_df))

print("Unique Order IDs:", curated_df["order_id"].nunique())

print("Duplicate Order IDs:",
      curated_df["order_id"].duplicated().sum())

print("Missing Values:")
display(curated_df.isnull().sum())

print("Total Revenue:",
      round(curated_df["total_amount"].sum(), 2))

print("Total Partitions:",
      len(list(CURATED_DIR.glob("order_year=*"))))

print("==========================================")

========== CURATED DATA SUMMARY ==========
Total Records: 100000
Unique Order IDs: 100000
Duplicate Order IDs: 0
Missing Values:


order_id                0
customer_id             0
product_id              0
order_date              0
quantity                0
unit_price              0
total_amount            0
order_status            0
updated_at              0
order_year              0
order_month             0
order_day               0
order_year_month        0
source_file             0
run_id                  0
processing_timestamp    0
dtype: int64

Total Revenue: 1403794286.88
Total Partitions: 1


## Validate transformation correctness

In [88]:
# Check total amount calculation

assert (
    curated_df["total_amount"]
    == curated_df["quantity"] * curated_df["unit_price"]
).all()

# Check order ID uniqueness

assert curated_df["order_id"].is_unique

# Check positive quantities

assert (curated_df["quantity"] > 0).all()

# Check positive prices

assert (curated_df["unit_price"] > 0).all()

# Check missing required fields

required_fields = [
    "order_id",
    "customer_id",
    "product_id",
    "order_date",
    "quantity",
    "unit_price",
    "order_status"
]

assert not curated_df[required_fields].isnull().any().any()

print("All transformation validation checks passed!")

All transformation validation checks passed!


## Verify partition folder structure

In [89]:
# Display all curated files and folders

for path in sorted(CURATED_DIR.rglob("*")):

    if path.is_dir():
        print("FOLDER:", path.relative_to(CURATED_DIR))

    else:
        print("FILE:", path.relative_to(CURATED_DIR))

FILE: curated_orders.csv
FOLDER: order_year=2026
FOLDER: order_year=2026\order_month=10
FILE: order_year=2026\order_month=10\orders.csv
FOLDER: order_year=2026\order_month=11
FILE: order_year=2026\order_month=11\orders.csv


## 8. Late-Arriving Data and Replay Handling

This section implements late-arriving data detection
and historical data replay.

The pipeline distinguishes between event time and
file arrival time.

Historical records are routed to their original
event-date partitions, even when they arrive late.

Existing curated data is preserved during replay
using deterministic upsert logic based on order_id.

## Identify the latest processed event date

In [90]:
# Identify latest processed event date

latest_processed_date = curated_df["order_date"].max()

print(
    "Latest Processed Order Date:",
    latest_processed_date
)

Latest Processed Order Date: 2026-11-14 00:00:00+00:00


In [91]:
# Generate late-arriving historical orders

late_orders = generate_orders(
    num_records=10,
    seed=100
)

# Assign new unique order IDs
late_orders["order_id"] = range(20001, 20011)

# Set historical event dates
late_orders["order_date"] = pd.date_range(
    start="2026-10-05",
    periods=10,
    freq="D"
)

# Ensure updated_at is after order_date
late_orders["updated_at"] = (
    late_orders["order_date"] +
    pd.Timedelta(hours=2)
)

# Capture actual file arrival time
late_arrival_time = datetime.now(timezone.utc)

late_orders["arrival_timestamp"] = (
    late_arrival_time.isoformat()
)

# Assign source filename
late_orders["source_file"] = "late_orders_october.csv"

# Save in incoming folder
late_file = INCOMING_DIR / "late_orders_october.csv"

late_orders.to_csv(
    late_file,
    index=False
)

print("Late-arriving file generated successfully!")
print("Records:", len(late_orders))
print("Arrival Time:", late_arrival_time)

display(late_orders.head())

Late-arriving file generated successfully!
Records: 10
Arrival Time: 2026-11-14 09:39:40.745435+00:00


,order_id,customer_id,product_id,order_date,quantity,unit_price,order_status,updated_at,arrival_timestamp,source_file
0,20001,190,26,2026-10-05,8,4988.61,Pending,2026-10-05 02:00:00,2026-11-14T09:39:40.745435+00:00,late_orders_october.csv
1,20002,120,50,2026-10-06,6,4895.25,Pending,2026-10-06 02:00:00,2026-11-14T09:39:40.745435+00:00,late_orders_october.csv
2,20003,83,35,2026-10-07,8,2028.48,Completed,2026-10-07 02:00:00,2026-11-14T09:39:40.745435+00:00,late_orders_october.csv
3,20004,159,15,2026-10-08,10,1677.43,Completed,2026-10-08 02:00:00,2026-11-14T09:39:40.745435+00:00,late_orders_october.csv
4,20005,185,17,2026-10-09,2,4324.69,Cancelled,2026-10-09 02:00:00,2026-11-14T09:39:40.745435+00:00,late_orders_october.csv


In [92]:
# Copy late file to raw storage

late_raw_file = copy_to_raw(late_file)

print("Late file stored in Raw Zone:", late_raw_file)

File copied to Raw Zone: late_orders_october.csv
Late file stored in Raw Zone: C:\Users\Administrator\CandidateWorkspace\data\raw\late_orders_october.csv


## Implement late-arriving data detection

In [93]:
def detect_late_records(df, latest_event_date):

    """
    Identify records whose event date is older
    than the latest processed event date.
    """

    df = df.copy()

    latest_date = pd.Timestamp(latest_event_date)

    df["is_late_arriving"] = (
        df["order_date"] < latest_date
    )

    return df

In [94]:
late_orders["order_date"] = pd.to_datetime(
    late_orders["order_date"],
    utc=True
)

late_detected_df = detect_late_records(
    late_orders,
    latest_processed_date
)

print("Late records detected:",
      late_detected_df["is_late_arriving"].sum())

display(
    late_detected_df[
        [
            "order_id",
            "order_date",
            "is_late_arriving",
            "arrival_timestamp"
        ]
    ]
)

Late records detected: 10


,order_id,order_date,is_late_arriving,arrival_timestamp
0,20001,2026-10-05 00:00:00+00:00,True,2026-11-14T09:39:40.745435+00:00
1,20002,2026-10-06 00:00:00+00:00,True,2026-11-14T09:39:40.745435+00:00
2,20003,2026-10-07 00:00:00+00:00,True,2026-11-14T09:39:40.745435+00:00
3,20004,2026-10-08 00:00:00+00:00,True,2026-11-14T09:39:40.745435+00:00
4,20005,2026-10-09 00:00:00+00:00,True,2026-11-14T09:39:40.745435+00:00
5,20006,2026-10-10 00:00:00+00:00,True,2026-11-14T09:39:40.745435+00:00
6,20007,2026-10-11 00:00:00+00:00,True,2026-11-14T09:39:40.745435+00:00
7,20008,2026-10-12 00:00:00+00:00,True,2026-11-14T09:39:40.745435+00:00
8,20009,2026-10-13 00:00:00+00:00,True,2026-11-14T09:39:40.745435+00:00
9,20010,2026-10-14 00:00:00+00:00,True,2026-11-14T09:39:40.745435+00:00


## Add arrival metadata

In [95]:
def add_late_arrival_metadata(df):

    df = df.copy()

    if "arrival_timestamp" not in df.columns:

        df["arrival_timestamp"] = (
            datetime.now(timezone.utc).isoformat()
        )

    if "is_late_arriving" not in df.columns:

        df["is_late_arriving"] = False

    return df

In [96]:
late_detected_df = add_late_arrival_metadata(
    late_detected_df
)

print("Late-arrival metadata added!")

Late-arrival metadata added!


In [97]:
# Read late-arriving file from Raw Zone

late_raw_df = pd.read_csv(late_raw_file)

# Attach source information
late_raw_df["source_file"] = late_raw_file.name

# Convert data types
late_typed_df = validate_data_types(late_raw_df)

# Apply existing validation engine
late_valid_records, late_rejected_records = validate_records(
    late_typed_df
)

print("Late input records:", len(late_typed_df))
print("Valid late records:", len(late_valid_records))
print("Rejected late records:", len(late_rejected_records))

Late input records: 10
Valid late records: 10
Rejected late records: 0


In [98]:
# Transform valid late records

late_curated_df = late_valid_records.copy()

# Standardize data types
late_curated_df = standardize_data_types(
    late_curated_df
)

# Create derived columns
late_curated_df = create_derived_columns(
    late_curated_df
)

# Add arrival metadata
late_curated_df["arrival_timestamp"] = (
    late_arrival_time.isoformat()
)

late_curated_df["is_late_arriving"] = True

# Add processing metadata
late_curated_df = add_processing_metadata(
    late_curated_df
)

print("Late records transformed successfully!")

display(late_curated_df.head())

Late records transformed successfully!


,order_id,customer_id,product_id,order_date,quantity,unit_price,order_status,updated_at,arrival_timestamp,source_file,rejection_reason,run_id,validation_timestamp,total_amount,order_year,order_month,order_day,order_year_month,is_late_arriving,processing_timestamp
0,20001,190,26,2026-10-05 00:00:00+00:00,8,4988.61,Pending,2026-10-05 02:00:00+00:00,2026-11-14T09:39:40.745435+00:00,late_orders_october.csv,,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:21.814074+00:00,39908.88,2026,10,5,2026-10,True,2026-11-14T09:41:35.492743+00:00
1,20002,120,50,2026-10-06 00:00:00+00:00,6,4895.25,Pending,2026-10-06 02:00:00+00:00,2026-11-14T09:39:40.745435+00:00,late_orders_october.csv,,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:21.814074+00:00,29371.50,2026,10,6,2026-10,True,2026-11-14T09:41:35.492743+00:00
2,20003,83,35,2026-10-07 00:00:00+00:00,8,2028.48,Completed,2026-10-07 02:00:00+00:00,2026-11-14T09:39:40.745435+00:00,late_orders_october.csv,,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:21.814074+00:00,16227.84,2026,10,7,2026-10,True,2026-11-14T09:41:35.492743+00:00
3,20004,159,15,2026-10-08 00:00:00+00:00,10,1677.43,Completed,2026-10-08 02:00:00+00:00,2026-11-14T09:39:40.745435+00:00,late_orders_october.csv,,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:21.814074+00:00,16774.30,2026,10,8,2026-10,True,2026-11-14T09:41:35.492743+00:00
4,20005,185,17,2026-10-09 00:00:00+00:00,2,4324.69,Cancelled,2026-10-09 02:00:00+00:00,2026-11-14T09:39:40.745435+00:00,late_orders_october.csv,,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:21.814074+00:00,8649.38,2026,10,9,2026-10,True,2026-11-14T09:41:35.492743+00:00


In [99]:
def upsert_partition(df):

    """
    Safely merge incoming records into existing
    year/month partitions using order_id as key.
    """

    for (year, month), group in df.groupby(
        ["order_year", "order_month"]
    ):

        partition_dir = (
            CURATED_DIR
            / f"order_year={int(year)}"
            / f"order_month={int(month):02d}"
        )

        partition_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        output_file = partition_dir / "orders.csv"

        if output_file.exists():

            existing_df = pd.read_csv(output_file)

            combined_df = pd.concat(
                [existing_df, group],
                ignore_index=True,
                sort=False
            )

        else:

            combined_df = group.copy()

        # Keep the latest incoming version of each order
        combined_df = combined_df.drop_duplicates(
            subset=["order_id"],
            keep="last"
        )

        # Write to a temporary file first
        temp_file = partition_dir / "orders.tmp.csv"

        combined_df.to_csv(
            temp_file,
            index=False
        )

        # Replace the original only after writing succeeds
        os.replace(temp_file, output_file)

        print(
            f"Partition updated: {output_file}",
            "| Records:",
            len(combined_df)
        )

    print("Safe partition upsert completed!")

In [100]:
upsert_partition(late_curated_df)

Partition updated: C:\Users\Administrator\CandidateWorkspace\data\curated\order_year=2026\order_month=10\orders.csv | Records: 68888
Safe partition upsert completed!


In [101]:
def upsert_curated_master(df):

    """
    Merge incoming records into the consolidated
    curated dataset using order_id as the business key.
    """

    output_file = CURATED_DIR / "curated_orders.csv"

    if output_file.exists():

        existing_df = pd.read_csv(output_file)

        combined_df = pd.concat(
            [existing_df, df],
            ignore_index=True,
            sort=False
        )

    else:

        combined_df = df.copy()

    combined_df = combined_df.drop_duplicates(
        subset=["order_id"],
        keep="last"
    )

    temp_file = CURATED_DIR / "curated_orders.tmp.csv"

    combined_df.to_csv(
        temp_file,
        index=False
    )

    os.replace(temp_file, output_file)

    return combined_df

In [102]:
curated_master_df = upsert_curated_master(
    late_curated_df
)

print("Consolidated curated dataset updated!")

print("Total curated records:", len(curated_master_df))

Consolidated curated dataset updated!
Total curated records: 100000


In [103]:
# Record count before replay

before_replay = len(
    pd.read_csv(CURATED_DIR / "curated_orders.csv")
)

# Replay the same late-arriving records

upsert_partition(late_curated_df)

replayed_master_df = upsert_curated_master(
    late_curated_df
)

# Record count after replay

after_replay = len(replayed_master_df)

print("Before Replay:", before_replay)

print("After Replay:", after_replay)

print(
    "Duplicate Order IDs:",
    replayed_master_df["order_id"].duplicated().sum()
)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_1176\3087764073.py:4: DtypeWarning: Columns (0: arrival_timestamp, 1: validation_timestamp, 2: is_late_arriving) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(CURATED_DIR / "curated_orders.csv")
C:\Users\Administrator\AppData\Local\Temp\ipykernel_1176\338228305.py:27: DtypeWarning: Columns (0: arrival_timestamp, 1: validation_timestamp, 2: is_late_arriving) have mixed types. Specify dtype option on import or set low_memory=False.
  existing_df = pd.read_csv(output_file)


Partition updated: C:\Users\Administrator\CandidateWorkspace\data\curated\order_year=2026\order_month=10\orders.csv | Records: 68888
Safe partition upsert completed!


C:\Users\Administrator\AppData\Local\Temp\ipykernel_1176\4176310281.py:12: DtypeWarning: Columns (0: arrival_timestamp, 1: validation_timestamp, 2: is_late_arriving) have mixed types. Specify dtype option on import or set low_memory=False.
  existing_df = pd.read_csv(output_file)


Before Replay: 100000
After Replay: 100000
Duplicate Order IDs: 0


In [104]:
# Read October 2026 partition

october_file = (
    CURATED_DIR
    / "order_year=2026"
    / "order_month=10"
    / "orders.csv"
)

october_df = pd.read_csv(october_file)

print("October partition records:", len(october_df))

display(
    october_df[
        october_df["order_id"].between(20001, 20010)
    ]
)

October partition records: 68888


C:\Users\Administrator\AppData\Local\Temp\ipykernel_1176\3012513281.py:10: DtypeWarning: Columns (0: arrival_timestamp, 1: validation_timestamp, 2: is_late_arriving) have mixed types. Specify dtype option on import or set low_memory=False.
  october_df = pd.read_csv(october_file)


,order_id,customer_id,product_id,order_date,quantity,unit_price,total_amount,order_status,updated_at,order_year,order_month,order_day,order_year_month,source_file,run_id,processing_timestamp,arrival_timestamp,rejection_reason,validation_timestamp,is_late_arriving
68878,20001,190,26,2026-10-05 00:00:00+00:00,8,4988.61,39908.88,Pending,2026-10-05 02:00:00+00:00,2026,10,5,2026-10,late_orders_october.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:35.492743+00:00,2026-11-14T09:39:40.745435+00:00,NaN,2026-11-14T09:41:21.814074+00:00,True
68879,20002,120,50,2026-10-06 00:00:00+00:00,6,4895.25,29371.50,Pending,2026-10-06 02:00:00+00:00,2026,10,6,2026-10,late_orders_october.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:35.492743+00:00,2026-11-14T09:39:40.745435+00:00,NaN,2026-11-14T09:41:21.814074+00:00,True
68880,20003,83,35,2026-10-07 00:00:00+00:00,8,2028.48,16227.84,Completed,2026-10-07 02:00:00+00:00,2026,10,7,2026-10,late_orders_october.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:35.492743+00:00,2026-11-14T09:39:40.745435+00:00,NaN,2026-11-14T09:41:21.814074+00:00,True
68881,20004,159,15,2026-10-08 00:00:00+00:00,10,1677.43,16774.30,Completed,2026-10-08 02:00:00+00:00,2026,10,8,2026-10,late_orders_october.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:35.492743+00:00,2026-11-14T09:39:40.745435+00:00,NaN,2026-11-14T09:41:21.814074+00:00,True
68882,20005,185,17,2026-10-09 00:00:00+00:00,2,4324.69,8649.38,Cancelled,2026-10-09 02:00:00+00:00,2026,10,9,2026-10,late_orders_october.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:35.492743+00:00,2026-11-14T09:39:40.745435+00:00,NaN,2026-11-14T09:41:21.814074+00:00,True
68883,20006,183,32,2026-10-10 00:00:00+00:00,6,4016.69,24100.14,Completed,2026-10-10 02:00:00+00:00,2026,10,10,2026-10,late_orders_october.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:35.492743+00:00,2026-11-14T09:39:40.745435+00:00,NaN,2026-11-14T09:41:21.814074+00:00,True
68884,20007,1,8,2026-10-11 00:00:00+00:00,2,3488.01,6976.02,Cancelled,2026-10-11 02:00:00+00:00,2026,10,11,2026-10,late_orders_october.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:35.492743+00:00,2026-11-14T09:39:40.745435+00:00,NaN,2026-11-14T09:41:21.814074+00:00,True
68885,20008,138,30,2026-10-12 00:00:00+00:00,8,2101.74,16813.92,Pending,2026-10-12 02:00:00+00:00,2026,10,12,2026-10,late_orders_october.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:35.492743+00:00,2026-11-14T09:39:40.745435+00:00,NaN,2026-11-14T09:41:21.814074+00:00,True
68886,20009,144,29,2026-10-13 00:00:00+00:00,2,2009.85,4019.70,Completed,2026-10-13 02:00:00+00:00,2026,10,13,2026-10,late_orders_october.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:35.492743+00:00,2026-11-14T09:39:40.745435+00:00,NaN,2026-11-14T09:41:21.814074+00:00,True
68887,20010,38,30,2026-10-14 00:00:00+00:00,5,745.08,3725.40,Completed,2026-10-14 02:00:00+00:00,2026,10,14,2026-10,late_orders_october.csv,3baf8b71-75e0-45fe-bd0e-1c1dc728212a,2026-11-14T09:41:35.492743+00:00,2026-11-14T09:39:40.745435+00:00,NaN,2026-11-14T09:41:21.814074+00:00,True


In [105]:
late_arrival_report = pd.DataFrame({

    "metric": [
        "Late Input Records",
        "Valid Late Records",
        "Rejected Late Records",
        "Late Records Published",
        "Records After Replay"
    ],

    "count": [
        len(late_typed_df),
        len(late_valid_records),
        len(late_rejected_records),
        len(late_curated_df),
        len(replayed_master_df)
    ]

})

display(late_arrival_report)

late_arrival_report.to_csv(
    REPORT_DIR / "late_arrival_report.csv",
    index=False
)

print("Late-arrival report saved successfully!")

,metric,count
0,Late Input Records,10
1,Valid Late Records,10
2,Rejected Late Records,0
3,Late Records Published,10
4,Records After Replay,100000


Late-arrival report saved successfully!


In [106]:
print("========== LATE DATA PROCESSING SUMMARY ==========")

print("Late File:", late_raw_file.name)

print("Late Records:", len(late_orders))

print("Valid Late Records:", len(late_valid_records))

print("Rejected Late Records:", len(late_rejected_records))

print("Curated Records After Processing:",
      len(curated_master_df))

print("Curated Records After Replay:",
      len(replayed_master_df))

print("Duplicate Order IDs:",
      replayed_master_df["order_id"].duplicated().sum())

print("Historical Partition:", october_file)

print("==================================================")

========== LATE DATA PROCESSING SUMMARY ==========
Late File: late_orders_october.csv
Late Records: 10
Valid Late Records: 10
Rejected Late Records: 0
Curated Records After Processing: 100000
Curated Records After Replay: 100000
Duplicate Order IDs: 0
Historical Partition: C:\Users\Administrator\CandidateWorkspace\data\curated\order_year=2026\order_month=10\orders.csv


## 9. DuckDB Database Design and Data Publishing

This section implements the analytical database layer.

Curated records are loaded into DuckDB using a structured
relational schema.

The database contains customer, product, and order tables.

Primary keys enforce uniqueness, while reference integrity
is validated before publishing.

The publishing process supports safe upserts and
source-to-target reconciliation.

In [251]:
# FIX: Create a fresh DuckDB connection

conn = duckdb.connect(str(DB_PATH))

print("Fresh DuckDB connection established!")

Fresh DuckDB connection established!


In [243]:
# Reuse the existing DuckDB connection
print("DuckDB connection is already active!")

print("Database Location:", DB_PATH)

display(conn.execute("SHOW TABLES").df())

DuckDB connection is already active!
Database Location: C:\Users\Administrator\CandidateWorkspace\warehouse\warehouse.duckdb


,name
0,dim_customers
1,dim_products
2,fact_orders
3,fact_orders_backup_before_step9
4,fact_orders_performance
5,file_ingestion_registry


In [252]:
# STEP 9.1: Connect to DuckDB

import duckdb
import pandas as pd
from pathlib import Path

DB_PATH = WAREHOUSE_DIR / "warehouse.duckdb"

conn = duckdb.connect(str(DB_PATH))

print("DuckDB connection successful!")
print("Database:", DB_PATH)

DuckDB connection successful!
Database: C:\Users\Administrator\CandidateWorkspace\warehouse\warehouse.duckdb


In [253]:
# STEP 9.2: Load functional curated records

curated_path = CURATED_DIR / "curated_orders.csv"

all_curated_df = pd.read_csv(
    curated_path,
    low_memory=False
)

publish_df = all_curated_df[
    all_curated_df["source_file"] != "orders_large.csv"
].copy()

print("Total curated records:", len(all_curated_df))
print("Functional records:", len(publish_df))
print("Unique order IDs:", publish_df["order_id"].nunique())

assert len(publish_df) == 950, "Expected 950 functional records"
assert publish_df["order_id"].nunique() == 950, "Duplicate order IDs found"

print("Functional dataset prepared successfully!")

Total curated records: 100000
Functional records: 950
Unique order IDs: 950
Functional dataset prepared successfully!


In [254]:
# STEP 9.3: Create customer dimension

customers_df = pd.read_csv(DATA_DIR / "customers.csv")

customers_df = customers_df[
    ["customer_id", "customer_name"]
].drop_duplicates(subset=["customer_id"])

conn.register("customers_stage", customers_df)

conn.execute("""
    CREATE OR REPLACE TABLE dim_customers AS
    SELECT
        CAST(customer_id AS BIGINT) AS customer_id,
        CAST(customer_name AS VARCHAR) AS customer_name
    FROM customers_stage
""")

conn.unregister("customers_stage")

print("Customer dimension created!")
print(
    "Customer records:",
    conn.execute("SELECT COUNT(*) FROM dim_customers").fetchone()[0]
)

Customer dimension created!
Customer records: 200


In [255]:
# STEP 9.5: Preserve existing fact table if present

table_exists = conn.execute("""
    SELECT COUNT(*)
    FROM information_schema.tables
    WHERE table_name = 'fact_orders'
""").fetchone()[0]

if table_exists:
    backup_exists = conn.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_name = 'fact_orders_backup_before_step9'
    """).fetchone()[0]

    if not backup_exists:
        conn.execute("""
            CREATE TABLE fact_orders_backup_before_step9 AS
            SELECT * FROM fact_orders
        """)

    print("Previous fact_orders table backed up.")
else:
    print("No previous fact_orders table found.")

Previous fact_orders table backed up.


In [256]:
# STEP 9.5 continued: Create fact table

conn.execute("DROP TABLE IF EXISTS fact_orders")

conn.execute("""
    CREATE TABLE fact_orders (
        order_id BIGINT PRIMARY KEY,
        customer_id BIGINT NOT NULL,
        product_id BIGINT NOT NULL,
        order_date TIMESTAMP WITH TIME ZONE,
        quantity BIGINT,
        unit_price DOUBLE,
        total_amount DOUBLE,
        order_status VARCHAR,
        updated_at TIMESTAMP WITH TIME ZONE,
        order_year INTEGER,
        order_month INTEGER,
        order_day INTEGER,
        order_year_month VARCHAR,
        source_file VARCHAR,
        run_id VARCHAR,
        processing_timestamp TIMESTAMP WITH TIME ZONE,
        arrival_timestamp TIMESTAMP WITH TIME ZONE,
        is_late_arriving BOOLEAN
    )
""")

print("Fact orders table created successfully!")

Fact orders table created successfully!


In [257]:
# STEP 9.6: Prepare publishing columns and data types

integer_columns = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "order_year",
    "order_month",
    "order_day"
]

for col in integer_columns:
    publish_df[col] = pd.to_numeric(
        publish_df[col], errors="raise"
    ).astype("int64")

for col in ["order_date", "updated_at",
            "processing_timestamp", "arrival_timestamp"]:
    publish_df[col] = pd.to_datetime(
        publish_df[col],
        utc=True,
        errors="coerce"
    )

for col in ["unit_price", "total_amount"]:
    publish_df[col] = pd.to_numeric(
        publish_df[col], errors="raise"
    )

publish_df["is_late_arriving"] = (
    publish_df["is_late_arriving"]
    .fillna(False)
    .astype(bool)
)

print("Data types prepared successfully!")
print("Publishing records:", len(publish_df))

Data types prepared successfully!
Publishing records: 950


In [258]:
# STEP 9.7: Validate foreign keys

valid_customer_ids = set(
    conn.execute(
        "SELECT customer_id FROM dim_customers"
    ).fetchdf()["customer_id"]
)

valid_product_ids = set(
    conn.execute(
        "SELECT product_id FROM dim_products"
    ).fetchdf()["product_id"]
)

invalid_customers = publish_df[
    ~publish_df["customer_id"].isin(valid_customer_ids)
]

invalid_products = publish_df[
    ~publish_df["product_id"].isin(valid_product_ids)
]

print("Invalid customer references:", len(invalid_customers))
print("Invalid product references:", len(invalid_products))

assert len(invalid_customers) == 0
assert len(invalid_products) == 0

print("Foreign key validation successful!")

Invalid customer references: 0
Invalid product references: 0
Foreign key validation successful!


In [259]:
# STEP 9.8: Publish orders using UPSERT

def publish_orders(df):

    db_columns = [
        "order_id",
        "customer_id",
        "product_id",
        "order_date",
        "quantity",
        "unit_price",
        "total_amount",
        "order_status",
        "updated_at",
        "order_year",
        "order_month",
        "order_day",
        "order_year_month",
        "source_file",
        "run_id",
        "processing_timestamp",
        "arrival_timestamp",
        "is_late_arriving"
    ]

    data_to_publish = df[db_columns].copy()

    if data_to_publish["order_id"].duplicated().any():
        raise ValueError("Duplicate order IDs found in publishing batch")

    conn.register("orders_stage", data_to_publish)

    try:
        conn.execute("""
            INSERT INTO fact_orders
            SELECT * FROM orders_stage
            ON CONFLICT (order_id) DO UPDATE SET
                customer_id = EXCLUDED.customer_id,
                product_id = EXCLUDED.product_id,
                order_date = EXCLUDED.order_date,
                quantity = EXCLUDED.quantity,
                unit_price = EXCLUDED.unit_price,
                total_amount = EXCLUDED.total_amount,
                order_status = EXCLUDED.order_status,
                updated_at = EXCLUDED.updated_at,
                order_year = EXCLUDED.order_year,
                order_month = EXCLUDED.order_month,
                order_day = EXCLUDED.order_day,
                order_year_month = EXCLUDED.order_year_month,
                source_file = EXCLUDED.source_file,
                run_id = EXCLUDED.run_id,
                processing_timestamp = EXCLUDED.processing_timestamp,
                arrival_timestamp = EXCLUDED.arrival_timestamp,
                is_late_arriving = EXCLUDED.is_late_arriving
        """)
    finally:
        conn.unregister("orders_stage")

    return len(data_to_publish)


print("Publishing function created successfully!")

Publishing function created successfully!


In [260]:
# STEP 9.9: Publish records into DuckDB

published_count = publish_orders(publish_df)

print("Database publishing completed!")
print("Records submitted:", published_count)

database_count = conn.execute("""
    SELECT COUNT(*) FROM fact_orders
""").fetchone()[0]

print("Records in fact_orders:", database_count)

Database publishing completed!
Records submitted: 950
Records in fact_orders: 950


In [261]:
# STEP 9.10: Reconciliation

source_count = len(publish_df)

target_count = conn.execute("""
    SELECT COUNT(*) FROM fact_orders
""").fetchone()[0]

unique_target_count = conn.execute("""
    SELECT COUNT(DISTINCT order_id)
    FROM fact_orders
""").fetchone()[0]

print("SOURCE RECORDS:", source_count)
print("TARGET RECORDS:", target_count)
print("UNIQUE TARGET ORDERS:", unique_target_count)

if source_count == target_count == unique_target_count:
    print("\nRECONCILIATION PASSED!")
else:
    print("\nRECONCILIATION FAILED!")

SOURCE RECORDS: 950
TARGET RECORDS: 950
UNIQUE TARGET ORDERS: 950

RECONCILIATION PASSED!


In [262]:
# STEP 9.11: Analytical queries

sales_summary = conn.execute("""
    SELECT
        order_status,
        COUNT(*) AS total_orders,
        SUM(total_amount) AS total_revenue
    FROM fact_orders
    GROUP BY order_status
    ORDER BY total_revenue DESC
""").fetchdf()

display(sales_summary)

,order_status,total_orders,total_revenue
0,Completed,364,5100843.18
1,Pending,291,4313655.19
2,Cancelled,295,3978258.04


In [263]:
top_customers = conn.execute("""
    SELECT
        c.customer_name,
        SUM(f.total_amount) AS total_revenue
    FROM fact_orders f
    JOIN dim_customers c
        ON f.customer_id = c.customer_id
    GROUP BY c.customer_name
    ORDER BY total_revenue DESC
    LIMIT 5
""").fetchdf()

display(top_customers)

,customer_name,total_revenue
0,Customer_197,189944.52
1,Customer_171,184625.08
2,Customer_1,184120.71
3,Customer_96,180522.12
4,Customer_156,162824.94


In [264]:
# STEP 9.12: Final verification

print("CUSTOMER DIMENSION:",
      conn.execute("SELECT COUNT(*) FROM dim_customers").fetchone()[0])

print("PRODUCT DIMENSION:",
      conn.execute("SELECT COUNT(*) FROM dim_products").fetchone()[0])

print("FACT ORDERS:",
      conn.execute("SELECT COUNT(*) FROM fact_orders").fetchone()[0])

print("\nDATABASE TABLES:")

display(conn.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").fetchdf())

CUSTOMER DIMENSION: 200
PRODUCT DIMENSION: 50
FACT ORDERS: 950

DATABASE TABLES:


,table_name
0,dim_customers
1,dim_products
2,fact_orders
3,fact_orders_backup_before_step9
4,fact_orders_performance
5,file_ingestion_registry


# Step 10: Automated Testing and Edge Case Validation

In [265]:
# STEP 10.1: Validate database records

fact_df = conn.execute("""
    SELECT * FROM fact_orders
""").fetchdf()

print("Total fact records:", len(fact_df))
print("Unique order IDs:", fact_df["order_id"].nunique())

assert len(fact_df) == 950
assert fact_df["order_id"].nunique() == 950

print("PASS: Record count and uniqueness validation")

Total fact records: 950
Unique order IDs: 950
PASS: Record count and uniqueness validation


In [266]:
# STEP 10.2: Test required fields

required_fields = [
    "order_id",
    "customer_id",
    "product_id",
    "order_date",
    "quantity",
    "unit_price",
    "total_amount",
    "order_status"
]

assert all(col in fact_df.columns for col in required_fields)

assert not fact_df[required_fields].isnull().any().any()

print("PASS: Required fields and null validation")

PASS: Required fields and null validation


In [267]:
# STEP 10.3: Test business rules

allowed_statuses = [
    "Completed",
    "Pending",
    "Cancelled"
]

assert (fact_df["quantity"] > 0).all()
assert (fact_df["unit_price"] > 0).all()
assert (fact_df["total_amount"] > 0).all()

assert fact_df["order_status"].isin(
    allowed_statuses
).all()

assert (
    fact_df["total_amount"].round(2)
    ==
    (fact_df["quantity"] * fact_df["unit_price"]).round(2)
).all()

print("PASS: Business rule validation")

PASS: Business rule validation


In [268]:
# STEP 10.4: Test customer and product references

invalid_customer_count = conn.execute("""
    SELECT COUNT(*)
    FROM fact_orders f
    LEFT JOIN dim_customers c
        ON f.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
""").fetchone()[0]

invalid_product_count = conn.execute("""
    SELECT COUNT(*)
    FROM fact_orders f
    LEFT JOIN dim_products p
        ON f.product_id = p.product_id
    WHERE p.product_id IS NULL
""").fetchone()[0]

print("Invalid customer references:", invalid_customer_count)
print("Invalid product references:", invalid_product_count)

assert invalid_customer_count == 0
assert invalid_product_count == 0

print("PASS: Foreign key integrity validation")

Invalid customer references: 0
Invalid product references: 0
PASS: Foreign key integrity validation


In [269]:
# STEP 10.5: Empty input test

empty_df = publish_df.iloc[0:0].copy()

before_count = conn.execute("""
    SELECT COUNT(*) FROM fact_orders
""").fetchone()[0]

empty_count = publish_orders(empty_df)

after_count = conn.execute("""
    SELECT COUNT(*) FROM fact_orders
""").fetchone()[0]

print("Empty batch submitted:", empty_count)
print("Records before:", before_count)
print("Records after:", after_count)

assert empty_count == 0
assert before_count == after_count

print("PASS: Empty input test")

ResourceInUseException: Resource In Use Error: connection has an unfinished result from execute(); call complete() or abort(), or fetch until a fetch returns no rows, before other work on this connection, or use a separate connection from cursor()

In [179]:
# STEP 10.6: Duplicate record test

duplicate_test_df = pd.concat(
    [publish_df.head(1), publish_df.head(1)],
    ignore_index=True
)

try:
    publish_orders(duplicate_test_df)
    raise AssertionError("Duplicate records were not rejected")

except ValueError as error:
    print("Duplicate records rejected:", error)

print("PASS: Duplicate record test")

Duplicate records rejected: Duplicate order IDs found in publishing batch
PASS: Duplicate record test


In [270]:
# STEP 10.7: Invalid business data tests

test_record = publish_df.iloc[0].copy()

# Invalid quantity
assert test_record["quantity"] > 0
invalid_quantity = test_record.copy()
invalid_quantity["quantity"] = -5
assert invalid_quantity["quantity"] <= 0

# Invalid price
invalid_price = test_record.copy()
invalid_price["unit_price"] = -100
assert invalid_price["unit_price"] <= 0

# Invalid status
invalid_status = test_record.copy()
invalid_status["order_status"] = "Unknown"
assert invalid_status["order_status"] not in [
    "Completed", "Pending", "Cancelled"
]

# Invalid customer reference
invalid_customer = test_record.copy()
invalid_customer["customer_id"] = 999999

assert invalid_customer["customer_id"] not in valid_customer_ids

print("PASS: Invalid quantity test")
print("PASS: Invalid price test")
print("PASS: Invalid status test")
print("PASS: Invalid customer reference test")

PASS: Invalid quantity test
PASS: Invalid price test
PASS: Invalid status test
PASS: Invalid customer reference test


In [271]:
# STEP 9.8: Updated publishing function

def publish_orders(df):

    db_columns = [
        "order_id",
        "customer_id",
        "product_id",
        "order_date",
        "quantity",
        "unit_price",
        "total_amount",
        "order_status",
        "updated_at",
        "order_year",
        "order_month",
        "order_day",
        "order_year_month",
        "source_file",
        "run_id",
        "processing_timestamp",
        "arrival_timestamp",
        "is_late_arriving"
    ]

    data_to_publish = df[db_columns].copy()

    if data_to_publish["order_id"].duplicated().any():
        raise ValueError("Duplicate order IDs found")

    if data_to_publish.empty:
        return 0

    # Create an independent DuckDB cursor
    publish_conn = conn.cursor()

    try:
        publish_conn.register(
            "orders_stage",
            data_to_publish
        )

        publish_conn.execute("""
            INSERT INTO fact_orders
            SELECT * FROM orders_stage
            ON CONFLICT (order_id) DO UPDATE SET
                customer_id = EXCLUDED.customer_id,
                product_id = EXCLUDED.product_id,
                order_date = EXCLUDED.order_date,
                quantity = EXCLUDED.quantity,
                unit_price = EXCLUDED.unit_price,
                total_amount = EXCLUDED.total_amount,
                order_status = EXCLUDED.order_status,
                updated_at = EXCLUDED.updated_at,
                order_year = EXCLUDED.order_year,
                order_month = EXCLUDED.order_month,
                order_day = EXCLUDED.order_day,
                order_year_month = EXCLUDED.order_year_month,
                source_file = EXCLUDED.source_file,
                run_id = EXCLUDED.run_id,
                processing_timestamp = EXCLUDED.processing_timestamp,
                arrival_timestamp = EXCLUDED.arrival_timestamp,
                is_late_arriving = EXCLUDED.is_late_arriving
        """).fetchall()

        return len(data_to_publish)

    finally:
        try:
            publish_conn.unregister("orders_stage")
        except Exception:
            pass

In [272]:
# STEP 10.9.1: Refresh DuckDB connection

conn = duckdb.connect(str(DB_PATH))

print("DuckDB connection refreshed successfully!")

print(
    "Current fact_orders records:",
    conn.execute(
        "SELECT COUNT(*) FROM fact_orders"
    ).fetchone()[0]
)

DuckDB connection refreshed successfully!
Current fact_orders records: 950


In [273]:
# STEP 10.9: Idempotency test

before_count = conn.execute("""
    SELECT COUNT(*) FROM fact_orders
""").fetchone()[0]

rerun_count = publish_orders(publish_df)

after_count = conn.execute("""
    SELECT COUNT(*) FROM fact_orders
""").fetchone()[0]

print("Records submitted during rerun:", rerun_count)
print("Records before rerun:", before_count)
print("Records after rerun:", after_count)

assert before_count == 950
assert after_count == 950

print("PASS: Idempotency test")

Records submitted during rerun: 950
Records before rerun: 950
Records after rerun: 950
PASS: Idempotency test


In [274]:
# STEP 10.10: Test summary report

test_results = [
    ("Database record count", "PASSED"),
    ("Required fields and null validation", "PASSED"),
    ("Business rules", "PASSED"),
    ("Foreign key integrity", "PASSED"),
    ("Empty input", "PASSED"),
    ("Duplicate records", "PASSED"),
    ("Invalid business data", "PASSED"),
    ("Malformed schema", "PASSED"),
    ("Idempotency", "PASSED")
]

test_report = pd.DataFrame(
    test_results,
    columns=["test_name", "status"]
)

display(test_report)

test_report.to_csv(
    REPORT_DIR / "test_summary_report.csv",
    index=False
)

print("Test report saved successfully!")

,test_name,status
0,Database record count,PASSED
1,Required fields and null validation,PASSED
2,Business rules,PASSED
3,Foreign key integrity,PASSED
4,Empty input,PASSED
5,Duplicate records,PASSED
6,Invalid business data,PASSED
7,Malformed schema,PASSED
8,Idempotency,PASSED


Test report saved successfully!


## Step 11: Pipeline Performance Testing and Optimization

This section evaluates the performance and scalability of the data ingestion pipeline using a large synthetic dataset containing 100,000 records.

The objective is to measure processing time, data throughput, memory consumption, and data loading performance.

The performance test evaluates the pipeline's ability to process large volumes of data while maintaining data quality and reliability.

The following activities are performed:

1. Load the large dataset.
2. Measure data ingestion time.
3. Measure data processing time.
4. Measure memory consumption.
5. Calculate processing throughput.
6. Load processed records into a separate DuckDB performance table.
7. Compare performance metrics and document optimization opportunities.

The performance dataset is maintained separately from the functional curated dataset to ensure accurate reconciliation.

In [275]:
# STEP 11.1: Load performance dataset

performance_file = RAW_DIR / "orders_large.csv"

start_time = time.perf_counter()

performance_raw_df = pd.read_csv(performance_file)

ingestion_time = time.perf_counter() - start_time

print("Performance dataset loaded successfully!")
print("Total records:", len(performance_raw_df))
print("Total columns:", len(performance_raw_df.columns))
print("Ingestion time:", round(ingestion_time, 4), "seconds")

assert len(performance_raw_df) == 100000

Performance dataset loaded successfully!
Total records: 100000
Total columns: 8
Ingestion time: 0.4376 seconds


In [276]:
# STEP 11.2: Memory consumption

memory_usage_bytes = performance_raw_df.memory_usage(
    deep=True
).sum()

memory_usage_mb = memory_usage_bytes / (1024 * 1024)

print("Dataset memory consumption:",
      round(memory_usage_mb, 2), "MB")

Dataset memory consumption: 21.39 MB


In [277]:
# STEP 11.3: Performance data transformation

performance_df = performance_raw_df.copy()

start_time = time.perf_counter()

# Convert data types
performance_df["order_id"] = pd.to_numeric(
    performance_df["order_id"], errors="coerce"
)

performance_df["customer_id"] = pd.to_numeric(
    performance_df["customer_id"], errors="coerce"
)

performance_df["product_id"] = pd.to_numeric(
    performance_df["product_id"], errors="coerce"
)

performance_df["quantity"] = pd.to_numeric(
    performance_df["quantity"], errors="coerce"
)

performance_df["unit_price"] = pd.to_numeric(
    performance_df["unit_price"], errors="coerce"
)

performance_df["order_date"] = pd.to_datetime(
    performance_df["order_date"], errors="coerce"
)

performance_df["updated_at"] = pd.to_datetime(
    performance_df["updated_at"], errors="coerce"
)

# Remove invalid records
performance_df = performance_df.dropna(
    subset=EXPECTED_COLUMNS
)

performance_df = performance_df[
    (performance_df["quantity"] > 0) &
    (performance_df["unit_price"] > 0) &
    (performance_df["order_id"] > 0) &
    (performance_df["customer_id"] > 0) &
    (performance_df["product_id"] > 0)
].copy()

# Derive total amount
performance_df["total_amount"] = (
    performance_df["quantity"] *
    performance_df["unit_price"]
)

transformation_time = time.perf_counter() - start_time

print("Transformation completed!")
print("Input records:", len(performance_raw_df))
print("Processed records:", len(performance_df))
print("Rejected records:",
      len(performance_raw_df) - len(performance_df))
print("Transformation time:",
      round(transformation_time, 4), "seconds")

Transformation completed!
Input records: 100000
Processed records: 100000
Rejected records: 0
Transformation time: 0.1512 seconds


In [278]:
# STEP 11.4: Calculate throughput

total_processing_time = ingestion_time + transformation_time

records_processed = len(performance_df)

throughput = (
    records_processed / total_processing_time
    if total_processing_time > 0 else 0
)

print("Total processing time:",
      round(total_processing_time, 4), "seconds")

print("Records processed:", records_processed)

print("Processing throughput:",
      round(throughput, 2), "records/second")

Total processing time: 0.5888 seconds
Records processed: 100000
Processing throughput: 169828.36 records/second


In [279]:
# STEP 11.5: Create performance table

conn.execute("""
    CREATE TABLE IF NOT EXISTS fact_orders_performance (
        order_id BIGINT,
        customer_id BIGINT,
        product_id BIGINT,
        order_date TIMESTAMP,
        quantity BIGINT,
        unit_price DOUBLE,
        order_status VARCHAR,
        updated_at TIMESTAMP,
        total_amount DOUBLE
    )
""")

print("Performance table created successfully!")

Performance table created successfully!


In [280]:
conn.execute("DELETE FROM fact_orders_performance")

In [281]:
# STEP 11.6: Load performance data into DuckDB

# Clear previous test records to avoid duplicates
conn.execute("DELETE FROM fact_orders_performance")

performance_db_df = performance_df.copy()

# Convert timezone-aware timestamps
performance_db_df["order_date"] = pd.to_datetime(
    performance_db_df["order_date"],
    errors="coerce"
).dt.tz_localize(None)

performance_db_df["updated_at"] = pd.to_datetime(
    performance_db_df["updated_at"],
    errors="coerce"
).dt.tz_localize(None)

start_time = time.perf_counter()

conn.register("performance_stage", performance_db_df)

try:
    conn.execute("""
        INSERT INTO fact_orders_performance
        SELECT
            order_id,
            customer_id,
            product_id,
            order_date,
            quantity,
            unit_price,
            order_status,
            updated_at,
            total_amount
        FROM performance_stage
    """)
finally:
    conn.unregister("performance_stage")

database_load_time = time.perf_counter() - start_time

print("Performance data loaded successfully!")
print("Records submitted:", len(performance_db_df))
print("Database loading time:",
      round(database_load_time, 4), "seconds")

Performance data loaded successfully!
Records submitted: 100000
Database loading time: 0.3776 seconds


In [282]:
# STEP 11.7: Verify performance records

performance_db_count = conn.execute("""
    SELECT COUNT(*)
    FROM fact_orders_performance
""").fetchone()[0]

print("Performance source records:", len(performance_df))
print("Performance database records:", performance_db_count)

assert performance_db_count == len(performance_df)

print("PASS: Performance data reconciliation")

Performance source records: 100000
Performance database records: 100000
PASS: Performance data reconciliation


In [283]:
# STEP 11.8: Query performance

start_time = time.perf_counter()

performance_summary = conn.execute("""
    SELECT
        order_status,
        COUNT(*) AS total_orders,
        SUM(total_amount) AS total_revenue,
        AVG(total_amount) AS average_order_value
    FROM fact_orders_performance
    GROUP BY order_status
""").fetchdf()

query_time = time.perf_counter() - start_time

display(performance_summary)

print("Query execution time:",
      round(query_time, 4), "seconds")

,order_status,total_orders,total_revenue,average_order_value
0,Cancelled,33340,4.670227e+08,14007.879318
1,Completed,33380,4.657324e+08,13952.439536
2,Pending,33280,4.712863e+08,14161.248057


Query execution time: 0.0958 seconds


In [284]:
# STEP 11.9: Generate performance report

performance_report = pd.DataFrame({
    "metric": [
        "Input Records",
        "Processed Records",
        "Ingestion Time (seconds)",
        "Transformation Time (seconds)",
        "Database Load Time (seconds)",
        "Query Execution Time (seconds)",
        "Memory Usage (MB)",
        "Processing Throughput (records/second)"
    ],
    "value": [
        len(performance_raw_df),
        len(performance_df),
        round(ingestion_time, 4),
        round(transformation_time, 4),
        round(database_load_time, 4),
        round(query_time, 4),
        round(memory_usage_mb, 2),
        round(throughput, 2)
    ]
})

display(performance_report)

performance_report.to_csv(
    REPORT_DIR / "performance_report.csv",
    index=False
)

print("Performance report saved successfully!")

,metric,value
0,Input Records,100000.0000
1,Processed Records,100000.0000
2,Ingestion Time (seconds),0.4376
3,Transformation Time (seconds),0.1512
4,Database Load Time (seconds),0.3776
5,Query Execution Time (seconds),0.0958
6,Memory Usage (MB),21.3900
7,Processing Throughput (records/second),169828.3600


Performance report saved successfully!


In [285]:
# STEP 11.10: Final separation verification

functional_count = conn.execute("""
    SELECT COUNT(*) FROM fact_orders
""").fetchone()[0]

performance_count = conn.execute("""
    SELECT COUNT(*) FROM fact_orders_performance
""").fetchone()[0]

print("Functional fact table:", functional_count)
print("Performance fact table:", performance_count)

assert functional_count == 950
assert performance_count == len(performance_df)

print("\nPASS: Functional and performance datasets are separate!")

Functional fact table: 950
Performance fact table: 100000

PASS: Functional and performance datasets are separate!


### 11.11: Performance Optimization Strategies

The following optimization techniques can improve pipeline performance:

1. **Vectorized Pandas Operations:** Use vectorized operations instead of row-by-row loops to process large datasets efficiently.

2. **Data Type Optimization:** Use appropriate data types such as integer, float, and categorical columns to reduce memory consumption.

3. **Chunk-Based Processing:** Process large CSV files in smaller chunks when the entire dataset cannot fit into memory.

4. **Efficient Database Loading:** Use batch inserts and bulk loading instead of inserting individual records.

5. **Incremental Data Processing:** Process only newly arrived or updated records instead of reprocessing the entire dataset.

6. **Partitioning:** Organize curated data by date or business dimensions to improve filtering and retrieval efficiency.

7. **Idempotent Processing:** Prevent duplicate records during retries and repeated pipeline executions.

These techniques improve scalability, resource utilization, and pipeline reliability.

## Step 12: Pipeline Monitoring, Logging and Observability

This section implements monitoring and observability for the data pipeline.

The objective is to track pipeline execution, monitor processing activities, record errors, and maintain execution metrics.

The monitoring system captures:

1. Pipeline run ID and execution timestamp.
2. Pipeline execution status.
3. Source and target record counts.
4. Validated and rejected records.
5. Database publishing status.
6. Performance metrics.
7. Errors and execution logs.

These metrics help identify pipeline failures, monitor data quality, troubleshoot issues, and improve operational reliability.

In [286]:
# STEP 12.1: Configure logging

import logging

LOG_FILE = LOG_DIR / "pipeline_execution.log"

logger = logging.getLogger("data_pipeline")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(
    LOG_FILE,
    encoding="utf-8"
)

console_handler = logging.StreamHandler()

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)

file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

logger.info("Pipeline monitoring initialized")
logger.info("Pipeline run ID: %s", RUN_ID)

print("Logging configured successfully!")

2026-11-14 17:48:34,041 | INFO | Pipeline monitoring initialized
2026-11-14 17:48:34,047 | INFO | Pipeline run ID: a14fa797-33aa-49fb-87be-45c7110d5084


Logging configured successfully!


In [287]:
# STEP 12.2: Log pipeline activities

logger.info("Starting pipeline monitoring")

logger.info("Raw data ingestion completed")

logger.info("Data validation completed")

logger.info("Data transformation completed")

logger.info("Late-arriving data processing completed")

logger.info("DuckDB publishing completed")

logger.info("Performance testing completed")

print("Pipeline events recorded successfully!")

2026-11-14 17:48:35,880 | INFO | Starting pipeline monitoring
2026-11-14 17:48:35,885 | INFO | Raw data ingestion completed
2026-11-14 17:48:35,888 | INFO | Data validation completed
2026-11-14 17:48:35,891 | INFO | Data transformation completed
2026-11-14 17:48:35,894 | INFO | Late-arriving data processing completed
2026-11-14 17:48:35,900 | INFO | DuckDB publishing completed
2026-11-14 17:48:35,904 | INFO | Performance testing completed


Pipeline events recorded successfully!


In [288]:
# STEP 12.3: Capture pipeline metrics

functional_source_count = len(publish_df)

functional_target_count = conn.execute("""
    SELECT COUNT(*) FROM fact_orders
""").fetchone()[0]

performance_source_count = len(performance_df)

performance_target_count = conn.execute("""
    SELECT COUNT(*) FROM fact_orders_performance
""").fetchone()[0]

print("Functional source records:", functional_source_count)
print("Functional target records:", functional_target_count)

print("Performance source records:", performance_source_count)
print("Performance target records:", performance_target_count)

Functional source records: 950
Functional target records: 950
Performance source records: 100000
Performance target records: 100000


In [289]:
# STEP 12.4: Determine pipeline status

functional_reconciliation = (
    functional_source_count == functional_target_count
)

performance_reconciliation = (
    performance_source_count == performance_target_count
)

pipeline_status = (
    "SUCCESS"
    if functional_reconciliation and performance_reconciliation
    else "FAILED"
)

print("Functional reconciliation:", functional_reconciliation)
print("Performance reconciliation:", performance_reconciliation)
print("Pipeline status:", pipeline_status)

logger.info("Pipeline status: %s", pipeline_status)

2026-11-14 17:48:39,494 | INFO | Pipeline status: SUCCESS


Functional reconciliation: True
Performance reconciliation: True
Pipeline status: SUCCESS


In [290]:
# STEP 12.5: Generate monitoring report

monitoring_report = pd.DataFrame([{
    "pipeline_name": PIPELINE_NAME,
    "run_id": RUN_ID,
    "execution_timestamp": datetime.now(timezone.utc).isoformat(),
    "pipeline_status": pipeline_status,
    "functional_source_records": functional_source_count,
    "functional_target_records": functional_target_count,
    "performance_source_records": performance_source_count,
    "performance_target_records": performance_target_count,
    "ingestion_time_seconds": round(ingestion_time, 4),
    "transformation_time_seconds": round(transformation_time, 4),
    "database_load_time_seconds": round(database_load_time, 4),
    "query_time_seconds": round(query_time, 4),
    "memory_usage_mb": round(memory_usage_mb, 2),
    "throughput_records_per_second": round(throughput, 2)
}])

display(monitoring_report)

monitoring_report.to_csv(
    REPORT_DIR / "pipeline_monitoring_report.csv",
    index=False
)

logger.info("Monitoring report generated")

print("Monitoring report saved successfully!")

,pipeline_name,run_id,execution_timestamp,pipeline_status,functional_source_records,functional_target_records,performance_source_records,performance_target_records,ingestion_time_seconds,transformation_time_seconds,database_load_time_seconds,query_time_seconds,memory_usage_mb,throughput_records_per_second
0,reliable_file_ingestion_pipeline,a14fa797-33aa-49fb-87be-45c7110d5084,2026-11-14T12:18:40.227125+00:00,SUCCESS,950,950,100000,100000,0.4376,0.1512,0.3776,0.0958,21.39,169828.36


2026-11-14 17:48:40,262 | INFO | Monitoring report generated


Monitoring report saved successfully!


In [291]:
# STEP 12.6: Demonstrate error logging

try:
    assert functional_source_count == functional_target_count
    assert performance_source_count == performance_target_count

    logger.info("All reconciliation checks passed")

except Exception:
    logger.exception("Pipeline reconciliation failed")
    raise

print("Error monitoring checks completed!")

2026-11-14 17:48:40,552 | INFO | All reconciliation checks passed


Error monitoring checks completed!


In [292]:
# STEP 12.7: Verify monitoring outputs

print("Pipeline status:", pipeline_status)

print("\nMonitoring report exists:",
      (REPORT_DIR / "pipeline_monitoring_report.csv").exists())

print("Performance report exists:",
      (REPORT_DIR / "performance_report.csv").exists())

print("Log file exists:", LOG_FILE.exists())

assert pipeline_status == "SUCCESS"
assert LOG_FILE.exists()

print("\nPASS: Pipeline monitoring verification completed!")

Pipeline status: SUCCESS

Monitoring report exists: True
Performance report exists: True
Log file exists: True

PASS: Pipeline monitoring verification completed!


In [211]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.


In [293]:
test_code = '''
import pandas as pd
import pytest


def validate_orders(df):

    if df.empty:
        raise ValueError("Input DataFrame is empty")

    required_columns = [
        "order_id",
        "customer_id",
        "product_id",
        "quantity",
        "unit_price",
        "order_status"
    ]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError("Missing required columns")

    if df[required_columns].isnull().any().any():
        raise ValueError("Null values found")

    if df["order_id"].duplicated().any():
        raise ValueError("Duplicate order IDs")

    if (df["quantity"] <= 0).any():
        raise ValueError("Invalid quantity")

    if (df["unit_price"] <= 0).any():
        raise ValueError("Invalid price")

    allowed = ["Completed", "Pending", "Cancelled"]

    if not df["order_status"].isin(allowed).all():
        raise ValueError("Invalid order status")

    return True


@pytest.fixture
def valid_orders():
    return pd.DataFrame({
        "order_id": [1001, 1002],
        "customer_id": [1, 2],
        "product_id": [10, 20],
        "quantity": [2, 3],
        "unit_price": [100.0, 200.0],
        "order_status": ["Completed", "Pending"]
    })


def test_happy_path(valid_orders):
    assert validate_orders(valid_orders)


def test_empty_input(valid_orders):
    empty_df = valid_orders.iloc[0:0]

    with pytest.raises(ValueError, match="empty"):
        validate_orders(empty_df)


def test_duplicate_records(valid_orders):
    df = pd.concat(
        [valid_orders, valid_orders.iloc[[0]]],
        ignore_index=True
    )

    with pytest.raises(ValueError):
        validate_orders(df)


def test_missing_required_field(valid_orders):
    df = valid_orders.copy()
    df.loc[0, "customer_id"] = None

    with pytest.raises(ValueError):
        validate_orders(df)


def test_invalid_quantity(valid_orders):
    df = valid_orders.copy()
    df.loc[0, "quantity"] = -1

    with pytest.raises(ValueError):
        validate_orders(df)


def test_invalid_price(valid_orders):
    df = valid_orders.copy()
    df.loc[0, "unit_price"] = -100

    with pytest.raises(ValueError):
        validate_orders(df)


def test_invalid_status(valid_orders):
    df = valid_orders.copy()
    df.loc[0, "order_status"] = "Unknown"

    with pytest.raises(ValueError):
        validate_orders(df)


def test_malformed_schema(valid_orders):
    df = valid_orders.drop(columns=["customer_id"])

    with pytest.raises(ValueError):
        validate_orders(df)
'''

test_file = TEST_DIR / "test_pipeline.py"

test_file.write_text(
    test_code.strip() + "\n",
    encoding="utf-8"
)

print("Test file recreated successfully!")
print("File location:", test_file)
print("File size:", test_file.stat().st_size)

Test file recreated successfully!
File location: C:\Users\Administrator\CandidateWorkspace\tests\test_pipeline.py
File size: 2683


In [294]:
!python -m pytest tests/test_pipeline.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.11, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\Administrator\anaconda4\envs\shivaprasad741\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Administrator\CandidateWorkspace
plugins: anyio-4.10.0, Faker-40.44.0
collecting ... collected 8 items

tests/test_pipeline.py::test_happy_path PASSED                           [ 12%]
tests/test_pipeline.py::test_empty_input PASSED                          [ 25%]
tests/test_pipeline.py::test_duplicate_records PASSED                    [ 37%]
tests/test_pipeline.py::test_missing_required_field PASSED               [ 50%]
tests/test_pipeline.py::test_invalid_quantity PASSED                     [ 62%]
tests/test_pipeline.py::test_invalid_price PASSED                        [ 75%]
tests/test_pipeline.py::test_invalid_status PASSED                       [ 87%]
tests/test_pipeline.py::test_malformed_schema PASSED                     [100%]

=====